# Kaggle Training Notebook

This notebook runs the full backbone-training workflow on Kaggle, saves reusable artifacts, and finishes with metric, loss, and SR-vs-LR comparisons.


## Strategy

1. Prepare the Kaggle environment and install the repo dependencies.
2. Cache backbone features for comparable mask and pooling strategies.
3. Benchmark deep-only, handcrafted-only, and concatenated representations with shared splits.
4. Run the classical and deep baseline sweeps when needed.
5. Load saved metrics and produce representation, pooling, mask, and SR-vs-LR comparisons.


In [ ]:
!pip install -q pandas seaborn xgboost lightgbm catboost torchgeo


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: c:\Users\ahtrabelsi\Desktop\stage\s2-super-resolution\.venv\Scripts\python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

repo_candidates = [
    Path.cwd(),
    Path("/kaggle/working/S2-super-resolution"),
    Path("/kaggle/working/s2-super-resolution"),
]
REPO = None
for cand in repo_candidates:
    if (cand / "pyproject.toml").exists() and (cand / "scripts").exists():
        REPO = cand
        break

if REPO is None:
    REPO = Path("/kaggle/working/S2-super-resolution")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/AhmedTrb/S2-super-resolution.git", str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "torchgeo"], check=True)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
sns.set_theme(style="whitegrid", context="talk")

print("Working directory:", REPO)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

ImportError: numpy._core.multiarray failed to import

: 

## Kaggle Setup

Clone the repository if needed, install dependencies, and confirm that CUDA is available before launching the experiment sweeps.


In [ ]:
# Core experiment declarations repaired after notebook corruption
python_bin = sys.executable
ML_ROOT = REPO / "outputs" / "yellowness_backbone_ml"
DEEP_ROOT = REPO / "outputs" / "yellowness_regression"
PLOT_ROOT = REPO / "outputs" / "kaggle_plots"
for path in [ML_ROOT, DEEP_ROOT, PLOT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_SPECS = [
    # CNN
    # ("torchgeo:resnet50", "ResNet50_Weights.SENTINEL2_ALL_DINO", "resnet50_dino"),
    ("torchgeo:resnet50", "ResNet50_Weights.SENTINEL2_SI_MS_SATLAS", "resnet50_satlas"),

    # Swin
    ("torchgeo:swin_v2_t", "Swin_V2_T_Weights.SENTINEL2_SI_MS_SATLAS", "swin_t_satlas"),
    # ("torchgeo:swin_v2_b", "Swin_V2_B_Weights.SENTINEL2_SI_MS_SATLAS", "swin_b_satlas"),

    # ViT
    ("torchgeo:vit_small_patch16_224", "ViTSmall16_Weights.SENTINEL2_ALL_DINO", "vit_small_dino"),
    # ("torchgeo:vit_base_patch16_224", "ViTBase16_Weights.SENTINEL2_ALL_MAE", "vit_base_mae"),
    # ("torchgeo:vit_base_patch14_dinov2", "ViTBase14_DINOv2_Weights.SENTINEL2_ALL_SOFTCON", "vit_base_softcon"),
]

CLASSIC_REGRESSORS = [
    "bayesian_ridge",
    "ridge",
    "elastic_net",
    "random_forest",
    "extra_trees",
    "hist_gradient_boosting",
    "xgboost",
]

SR_EXPERIMENTS = [
    {
        "resolution": "sr",
        "backbone": backbone,
        "weight": weight,
        "run_name": f"sr_{short_name}",
    }
    for backbone, weight, short_name in MODEL_SPECS
]

LR_EXPERIMENTS = [
    {
        "resolution": "lr",
        "backbone": backbone,
        "weight": weight,
        "run_name": f"lr_{short_name}",
    }
    for backbone, weight, short_name in MODEL_SPECS
]

print("Active model families:")
for _, _, short_name in MODEL_SPECS:
    print(" -", short_name)

## Classical Backbone ML Sweep

This section runs the feature-extraction plus classical regression sweep for the full SR and LR experiment matrix.


In [ ]:
# Run the classical backbone-ML sweep end-to-end.
# Set RUN_FULL_SWEEP = False if you only want to inspect the configuration first.
RUN_FULL_SWEEP = True


def run_backbone_ml_experiments(experiments):
    for spec in experiments:
        out_dir = ML_ROOT / spec["run_name"]
        cmd = [
            python_bin,
            "scripts/train_yellowness_backbone_ml.py",
            "--inventory", str(INVENTORY),
            "--root-dir", str(DATASET_DIR),
            "--resolution", spec["resolution"],
            "--backbone", spec["backbone"],
            "--torchgeo-weight", spec["weight"],
            "--mask-fusion", "feature_mask_pool",
            "--pooling", "global_avg",
            "--no-data-parallel",
            "--regressors", *CLASSIC_REGRESSORS,
            "--dr-methods", "none", "pca",
            "--pca-variance", "0.95",
            "--pls-top-k", "8",
            "--batch-size", "16",
            "--num-workers", "4",
            "--save-feature-csv",
            "--feature-csv-name", f"{spec['run_name']}_embeddings.csv",
            "--export-deep-runs-summary",
            "--deep-runs-root", str(DEEP_ROOT),
            "--output-dir", str(out_dir),
        ]
        run_cmd(cmd)


if RUN_FULL_SWEEP:
    run_backbone_ml_experiments(SR_EXPERIMENTS)
    run_backbone_ml_experiments(LR_EXPERIMENTS)


In [ ]:
# Leakage-free grouped CV benchmark for spectral + frozen deep features
from pathlib import Path
from datetime import datetime
import json
import shutil
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import FileLink, display
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
 )
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

sns.set_theme(style="whitegrid", context="talk")

# ---------------------------------------------------------------------
# 1) Standalone configuration
# ---------------------------------------------------------------------
repo_candidates = [
    Path.cwd(),
    Path("/kaggle/working/S2-super-resolution"),
    Path("/kaggle/working/s2-super-resolution"),
]
REPO = next((cand for cand in repo_candidates if (cand / "pyproject.toml").exists() and (cand / "scripts").exists()), Path.cwd())
INVENTORY = REPO / "yellowness_dataset" / "observations_inventory.csv"
DATASET_DIR = REPO / "yellowness_dataset"
PYTHON_BIN = sys.executable

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = REPO / "outputs" / "leakage_free_grouped_cv" / f"run_{RUN_TAG}"
RESULT_ROOT = RUN_ROOT / "results"
PLOT_ROOT = RUN_ROOT / "plots"
for path in [RUN_ROOT, RESULT_ROOT, PLOT_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

TARGET = "yellowness"
GROUP_COL = "id_plot"
ROW_COL = "row_index"
CV_FOLDS = 5
RANDOM_STATE = 42
CORR_THRESHOLD = 0.98
TOP_K_VALUES = [30, 40, 50]
MAX_PCA_COMPONENTS = 50
PERMUTATION_REPEATS = 8
REEXTRACT_IF_MISSING = False

SEVERITY_BINS = [-np.inf, 10, 60, np.inf]
SEVERITY_LABELS = ["low", "medium", "high"]

MODEL_SPECS = [
    {
        "family": "resnet",
        "backbone": "torchgeo:resnet50",
        "weight": "ResNet50_Weights.SENTINEL2_SI_MS_SATLAS",
        "run_name": "lr_resnet50_satlas",
        "match_terms": ["resnet50", "satlas"],
    },
    {
        "family": "swin",
        "backbone": "torchgeo:swin_v2_t",
        "weight": "Swin_V2_T_Weights.SENTINEL2_SI_MS_SATLAS",
        "run_name": "lr_swin_t_satlas",
        "match_terms": ["swin", "satlas"],
    },
    {
        "family": "vit",
        "backbone": "torchgeo:vit_small_patch16_224",
        "weight": "ViTSmall16_Weights.SENTINEL2_ALL_DINO",
        "run_name": "lr_vit_small_dino",
        "match_terms": ["vit", "small", "dino"],
    },
]

REGRESSOR_BUILDERS = {
    "ridge": lambda: Pipeline([
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 13))),
    ]),
    "elastic_net": lambda: Pipeline([
        ("scale", StandardScaler()),
        (
            "model",
            ElasticNetCV(
                alphas=np.logspace(-4, 1, 12),
                l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
                cv=3,
                random_state=RANDOM_STATE,
                max_iter=100000,
            ),
        ),
    ]),
    "random_forest": lambda: RandomForestRegressor(
        n_estimators=240,
        min_samples_leaf=2,
        max_features=0.7,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "svr": lambda: Pipeline([
        ("scale", StandardScaler()),
        ("model", SVR(C=10.0, epsilon=0.05, gamma="scale")),
    ]),
}

OPTIONAL_REGRESSOR_BUILDERS = {
    "xgboost": ("xgboost", "XGBRegressor"),
    "lightgbm": ("lightgbm", "LGBMRegressor"),
}

TREE_REGRESSORS = {"random_forest", "xgboost", "lightgbm"}

# ---------------------------------------------------------------------
# 2) Helpers
# ---------------------------------------------------------------------
def run_cmd(cmd, cwd=None, allow_failure=False):
    print("RUN:", " ".join(map(str, cmd)))
    try:
        subprocess.run([str(x) for x in cmd], check=True, cwd=str(cwd) if cwd else None)
        return True
    except subprocess.CalledProcessError as exc:
        print(f"FAILED: {exc}")
        if not allow_failure:
            raise
        return False


def metric_dict(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def severity_from_values(values):
    clipped = np.clip(np.asarray(values, dtype=float), 0.0, 100.0)
    return pd.cut(
        clipped,
        bins=SEVERITY_BINS,
        labels=SEVERITY_LABELS,
        include_lowest=True,
        right=True,
    ).astype(str)


def severity_metric_dict(y_true, y_pred):
    sev_true = severity_from_values(y_true)
    sev_pred = severity_from_values(y_pred)
    return {
        "severity_accuracy": float(accuracy_score(sev_true, sev_pred)),
        "severity_balanced_accuracy": float(balanced_accuracy_score(sev_true, sev_pred)),
        "severity_f1_macro": float(f1_score(sev_true, sev_pred, average="macro", zero_division=0)),
    }


def drop_high_corr(df_in, columns, threshold=0.98):
    if len(columns) <= 1:
        return list(columns)
    corr_mat = df_in[columns].corr(numeric_only=True).abs()
    upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return [col for col in columns if col not in to_drop]


def discover_embedding_csv(search_roots, spec):
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        candidates.extend(root.glob("**/*embeddings.csv"))
        candidates.extend(root.glob("**/backbone_embeddings.csv"))

    scored = []
    for path in candidates:
        text = str(path).lower()
        score = 0
        if spec["run_name"].lower() in text:
            score += 10
        score += sum(term in text for term in spec["match_terms"])
        if score > 0:
            scored.append((score, len(text), path))

    if not scored:
        return None

    scored.sort(key=lambda item: (-item[0], item[1], str(item[2])))
    return scored[0][2]


def load_embedding_table(spec):
    search_roots = [
        REPO / "outputs" / "yellowness_backbone_ml",
        Path("/kaggle/working") / "outputs" / "yellowness_backbone_ml",
    ]
    csv_path = discover_embedding_csv(search_roots, spec)

    if csv_path is None and REEXTRACT_IF_MISSING:
        out_dir = RUN_ROOT / "reextracted_embeddings" / spec["run_name"]
        out_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            PYTHON_BIN,
            "scripts/train_yellowness_backbone_ml.py",
            "--inventory", str(INVENTORY),
            "--root-dir", str(DATASET_DIR),
            "--resolution", "lr",
            "--backbone", spec["backbone"],
            "--torchgeo-weight", spec["weight"],
            "--mask-fusion", "feature_mask_pool",
            "--pooling", "global_avg",
            "--no-data-parallel",
            "--batch-size", "16",
            "--num-workers", "4",
            "--extract-only",
            "--feature-csv-name", "backbone_embeddings.csv",
            "--output-dir", str(out_dir),
        ]
        ok = run_cmd(cmd, cwd=REPO, allow_failure=True)
        extracted_csv = out_dir / "backbone_embeddings.csv"
        if ok and extracted_csv.exists() and extracted_csv.stat().st_size > 0:
            csv_path = extracted_csv

    if csv_path is None:
        raise FileNotFoundError(f"No embedding CSV found for {spec['family']} using run name {spec['run_name']}")

    emb_df = pd.read_csv(csv_path)
    feat_cols = [col for col in emb_df.columns if col.startswith("feat_")]
    if ROW_COL not in emb_df.columns or not feat_cols:
        raise ValueError(f"Embedding file missing {ROW_COL} or feat_* columns: {csv_path}")

    emb_df = emb_df.dropna(subset=[ROW_COL]).copy()
    emb_df[ROW_COL] = emb_df[ROW_COL].astype(int)
    if emb_df[ROW_COL].duplicated().any():
        emb_df = emb_df.groupby(ROW_COL, as_index=False)[feat_cols].mean()
    else:
        emb_df = emb_df[[ROW_COL, *feat_cols]].copy()

    renamed = {col: f"{spec['family']}__{col}" for col in feat_cols}
    emb_df = emb_df.rename(columns=renamed)
    return csv_path, emb_df, list(renamed.values())


def resolve_group_strata(df, requested_folds):
    group_target = df.groupby(GROUP_COL)[TARGET].mean()
    if group_target.empty:
        raise RuntimeError("No parcel groups found after merging embeddings.")

    for q in range(min(5, int(group_target.nunique())), 1, -1):
        try:
            bins = pd.qcut(group_target.rank(method="first"), q=q, labels=False, duplicates="drop")
        except Exception:
            continue
        bins = pd.Series(bins, index=group_target.index).astype(int)
        counts = bins.value_counts()
        folds = min(requested_folds, int(counts.min())) if not counts.empty else 0
        if folds >= 2:
            return bins, folds, f"quantile_{len(counts)}"

    severity_bins = pd.cut(
        group_target,
        bins=SEVERITY_BINS,
        labels=False,
        include_lowest=True,
        right=True,
    )
    severity_bins = pd.Series(severity_bins, index=group_target.index).fillna(0).astype(int)
    counts = severity_bins.value_counts()
    folds = min(requested_folds, int(counts.min())) if not counts.empty else 0
    if folds >= 2:
        return severity_bins, folds, "severity_fallback"

    raise RuntimeError("Unable to build at least 2 stratified parcel folds from current data.")


def build_optional_regressor(name):
    module_name, class_name = OPTIONAL_REGRESSOR_BUILDERS[name]
    try:
        module = __import__(module_name, fromlist=[class_name])
        cls = getattr(module, class_name)
    except Exception:
        return None

    if name == "xgboost":
        return cls(
            n_estimators=180,
            max_depth=5,
            learning_rate=0.04,
            subsample=0.9,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=4,
        )
    return None


def build_regressors():
    models = {name: builder() for name, builder in REGRESSOR_BUILDERS.items()}
    for name in OPTIONAL_REGRESSOR_BUILDERS:
        model = build_optional_regressor(name)
        if model is not None:
            models[name] = model
        else:
            warnings.warn(f"Optional regressor unavailable and will be skipped: {name}")
    return models


def prepare_feature_block(train_df, val_df, feature_cols, y_train, selection_mode, selection_value=None):
    cols = [col for col in feature_cols if col in train_df.columns]
    if not cols:
        return None

    x_train_df = train_df[cols].copy()
    x_val_df = val_df[cols].copy()

    train_medians = x_train_df.median(numeric_only=True)
    x_train_df = x_train_df.replace([np.inf, -np.inf], np.nan).fillna(train_medians)
    x_val_df = x_val_df.replace([np.inf, -np.inf], np.nan).fillna(train_medians)

    valid_cols = [col for col in x_train_df.columns if x_train_df[col].nunique(dropna=True) > 1]
    if not valid_cols:
        return None

    x_train_df = x_train_df[valid_cols]
    x_val_df = x_val_df[valid_cols]
    decor_cols = drop_high_corr(x_train_df, list(x_train_df.columns), threshold=CORR_THRESHOLD)
    if not decor_cols:
        return None

    x_train_df = x_train_df[decor_cols]
    x_val_df = x_val_df[decor_cols]

    if selection_mode == "mi":
        top_k = min(int(selection_value), x_train_df.shape[1])
        if top_k < 1:
            return None
        mi_scores = mutual_info_regression(x_train_df, y_train, random_state=RANDOM_STATE)
        mi_series = pd.Series(mi_scores, index=x_train_df.columns).sort_values(ascending=False)
        selected_cols = mi_series.head(top_k).index.tolist()
        return {
            "X_train": x_train_df[selected_cols].to_numpy(dtype=np.float32),
            "X_val": x_val_df[selected_cols].to_numpy(dtype=np.float32),
            "feature_names": selected_cols,
            "selection_label": f"mi_top_{top_k:02d}",
            "decorrelated_dim": int(x_train_df.shape[1]),
            "final_dim": int(len(selected_cols)),
        }

    if selection_mode == "pca95":
        scaler = StandardScaler()
        x_train_sc = scaler.fit_transform(x_train_df)
        x_val_sc = scaler.transform(x_val_df)

        full_pca = PCA(random_state=RANDOM_STATE)
        full_pca.fit(x_train_sc)
        cumvar = np.cumsum(full_pca.explained_variance_ratio_)
        needed = int(np.searchsorted(cumvar, 0.95) + 1)
        n_components = max(1, min(needed, MAX_PCA_COMPONENTS, x_train_sc.shape[0] - 1, x_train_sc.shape[1]))

        reducer = PCA(n_components=n_components, random_state=RANDOM_STATE)
        x_train_pca = reducer.fit_transform(x_train_sc)
        x_val_pca = reducer.transform(x_val_sc)
        feature_names = [f"pc_{idx + 1:02d}" for idx in range(x_train_pca.shape[1])]
        return {
            "X_train": x_train_pca.astype(np.float32),
            "X_val": x_val_pca.astype(np.float32),
            "feature_names": feature_names,
            "selection_label": f"pca95_cap_{x_train_pca.shape[1]:02d}",
            "decorrelated_dim": int(x_train_df.shape[1]),
            "final_dim": int(x_train_pca.shape[1]),
        }

    raise ValueError(f"Unsupported selection mode: {selection_mode}")


def aggregate_mean_std(frame, group_cols, metric_cols):
    grouped = frame.groupby(group_cols, dropna=False)[metric_cols].agg(["mean", "std"]).reset_index()
    grouped.columns = [
        "_".join([str(part) for part in col if part]).rstrip("_")
        for col in grouped.columns.to_flat_index()
    ]
    for metric in metric_cols:
        grouped[f"{metric}_mean_std"] = grouped.apply(
            lambda row: f"{row[f'{metric}_mean']:.4f} ± {0.0 if pd.isna(row[f'{metric}_std']) else row[f'{metric}_std']:.4f}",
            axis=1,
        )
    return grouped


# ---------------------------------------------------------------------
# 3) Load inventory + cached embeddings
# ---------------------------------------------------------------------
if not INVENTORY.exists():
    raise FileNotFoundError(f"Inventory not found: {INVENTORY}")

inventory_df = pd.read_csv(INVENTORY).reset_index(drop=True)
inventory_df[ROW_COL] = inventory_df.index.astype(int)

excluded_exact = {TARGET, GROUP_COL, ROW_COL, "year", "parcel_index", "source_file", "source_inventory"}
spectral_prefixes = ("lr_", "sr_", "calc_lr_", "calc_sr_")
meta_features = [col for col in ["month", "day_of_year", "s2_date_difference_days"] if col in inventory_df.columns]
spectral_features = [
    col
    for col in inventory_df.columns
    if pd.api.types.is_numeric_dtype(inventory_df[col])
    and col not in excluded_exact
    and inventory_df[col].nunique(dropna=True) > 1
    and (col.startswith(spectral_prefixes) or col in meta_features)
]

merged_df = inventory_df[[ROW_COL, GROUP_COL, TARGET, *sorted(set(spectral_features))]].copy()
deep_feature_sets = {}
embedding_sources = {}

for spec in MODEL_SPECS:
    csv_path, emb_df, deep_cols = load_embedding_table(spec)
    embedding_sources[spec["family"]] = str(csv_path)
    deep_feature_sets[spec["family"]] = deep_cols
    merged_df = merged_df.merge(emb_df, on=ROW_COL, how="inner")

if merged_df.empty:
    raise RuntimeError("No rows remain after aligning inventory with the three embedding tables.")
if merged_df[GROUP_COL].nunique() < 2:
    raise RuntimeError("Need at least two parcels after alignment to run grouped CV.")

print(f"Aligned rows: {len(merged_df)}")
print(f"Aligned parcels: {merged_df[GROUP_COL].nunique()}")
print("Embedding sources:")
for family, path in embedding_sources.items():
    print(f" - {family}: {path}")

feature_sets = {
    "spectral_only": spectral_features,
    "resnet_only": deep_feature_sets["resnet"],
    "resnet152_only": deep_feature_sets["resnet152"],
    "swin_only": deep_feature_sets["swin"],
    "swin_b_only": deep_feature_sets["swin_b"],
    "resnet_plus_spectral": deep_feature_sets["resnet"] + spectral_features,
    "resnet152_plus_spectral": deep_feature_sets["resnet152"] + spectral_features,
    "swin_plus_spectral": deep_feature_sets["swin"] + spectral_features,
    "swin_b_plus_spectral": deep_feature_sets["swin_b"] + spectral_features,
}

group_strata, effective_folds, strat_source = resolve_group_strata(merged_df, CV_FOLDS)
row_strata = merged_df[GROUP_COL].map(group_strata).astype(int)
print(f"Using {effective_folds} grouped folds with strata source: {strat_source}")
print(group_strata.value_counts().sort_index().to_string())

cv = StratifiedGroupKFold(n_splits=effective_folds, shuffle=True, random_state=RANDOM_STATE)
regressors = build_regressors()
selection_configs = [("mi", top_k) for top_k in TOP_K_VALUES] + [("pca95", MAX_PCA_COMPONENTS)]

all_rows = []
prediction_rows = []
importance_rows = []

# ---------------------------------------------------------------------
# 4) Leakage-free grouped CV
# ---------------------------------------------------------------------
for fold_id, (train_idx, val_idx) in enumerate(cv.split(merged_df, row_strata, groups=merged_df[GROUP_COL]), start=1):
    train_df = merged_df.iloc[train_idx].copy()
    val_df = merged_df.iloc[val_idx].copy()

    train_groups = set(train_df[GROUP_COL].astype(str))
    val_groups = set(val_df[GROUP_COL].astype(str))
    if train_groups & val_groups:
        raise RuntimeError("Grouped CV leakage detected: at least one parcel appears in both train and validation.")

    y_train = train_df[TARGET].to_numpy(dtype=np.float32)
    y_val = val_df[TARGET].to_numpy(dtype=np.float32)

    print(f"\nFold {fold_id}/{effective_folds} | train rows={len(train_df)} | val rows={len(val_df)} | train parcels={train_df[GROUP_COL].nunique()} | val parcels={val_df[GROUP_COL].nunique()}")

    for feature_set_name, feature_cols in feature_sets.items():
        for selection_mode, selection_value in selection_configs:
            prepared = prepare_feature_block(train_df, val_df, feature_cols, y_train, selection_mode, selection_value)
            if prepared is None or prepared["final_dim"] < 1:
                continue

            x_train = prepared["X_train"]
            x_val = prepared["X_val"]
            feature_names = prepared["feature_names"]

            for reg_name, regressor in regressors.items():
                model = clone(regressor)
                fit_kwargs = {}
                if reg_name == "xgboost":
                    fit_kwargs = {"eval_set": [(x_val, y_val)], "verbose": False}
                if reg_name == "lightgbm":
                    fit_kwargs = {"eval_set": [(x_val, y_val)]}

                try:
                    model.fit(x_train, y_train, **fit_kwargs)
                    pred_val = np.asarray(model.predict(x_val)).reshape(-1)
                except Exception as exc:
                    warnings.warn(
                        f"Skipping {feature_set_name} | {prepared['selection_label']} | {reg_name} on fold {fold_id}: {exc}"
                    )
                    continue

                reg_metrics = metric_dict(y_val, pred_val)
                sev_metrics = severity_metric_dict(y_val, pred_val)

                all_rows.append({
                    "fold": fold_id,
                    "feature_set": feature_set_name,
                    "selection_mode": selection_mode,
                    "selection_value": selection_value,
                    "selection_label": prepared["selection_label"],
                    "regressor": reg_name,
                    "train_rows": int(len(train_df)),
                    "val_rows": int(len(val_df)),
                    "train_parcels": int(train_df[GROUP_COL].nunique()),
                    "val_parcels": int(val_df[GROUP_COL].nunique()),
                    "decorrelated_dim": prepared["decorrelated_dim"],
                    "feature_dim": prepared["final_dim"],
                    **reg_metrics,
                    **sev_metrics,
                })

                pred_frame = pd.DataFrame({
                    ROW_COL: val_df[ROW_COL].to_numpy(),
                    GROUP_COL: val_df[GROUP_COL].to_numpy(),
                    "fold": fold_id,
                    "feature_set": feature_set_name,
                    "selection_label": prepared["selection_label"],
                    "regressor": reg_name,
                    "y_true": y_val,
                    "y_pred": pred_val,
                    "severity_true": severity_from_values(y_val),
                    "severity_pred": severity_from_values(pred_val),
                })
                prediction_rows.append(pred_frame)

                if reg_name in TREE_REGRESSORS and len(feature_names) <= 50:
                    try:
                        perm = permutation_importance(
                            model,
                            x_val,
                            y_val,
                            n_repeats=PERMUTATION_REPEATS,
                            random_state=RANDOM_STATE,
                            scoring="neg_root_mean_squared_error",
                            n_jobs=1,
                        )
                        for feat_name, mean_imp, std_imp in zip(feature_names, perm.importances_mean, perm.importances_std):
                            importance_rows.append({
                                "fold": fold_id,
                                "feature_set": feature_set_name,
                                "selection_label": prepared["selection_label"],
                                "regressor": reg_name,
                                "feature": feat_name,
                                "importance_mean": float(mean_imp),
                                "importance_std": float(std_imp),
                            })
                    except Exception as exc:
                        warnings.warn(
                            f"Permutation importance failed for {feature_set_name} | {prepared['selection_label']} | {reg_name} on fold {fold_id}: {exc}"
                        )

# ---------------------------------------------------------------------
# 5) Save results
# ---------------------------------------------------------------------
fold_results = pd.DataFrame(all_rows)
if fold_results.empty:
    raise RuntimeError("No successful model fits were produced by the grouped CV benchmark.")

predictions_df = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
importance_df = pd.DataFrame(importance_rows)

fold_results = fold_results.sort_values(["rmse", "r2", "mae"], ascending=[True, False, True]).reset_index(drop=True)
fold_results.to_csv(RESULT_ROOT / "fold_metrics.csv", index=False)
fold_results.to_json(RESULT_ROOT / "fold_metrics.json", orient="records", indent=2)

if not predictions_df.empty:
    predictions_df.to_csv(RESULT_ROOT / "fold_predictions.csv", index=False)

summary = aggregate_mean_std(
    fold_results,
    ["feature_set", "selection_label", "regressor"],
    ["rmse", "mae", "r2", "severity_accuracy", "severity_balanced_accuracy", "severity_f1_macro"],
)
summary = summary.sort_values(["rmse_mean", "r2_mean", "mae_mean"], ascending=[True, False, True]).reset_index(drop=True)
summary.to_csv(RESULT_ROOT / "summary_mean_std.csv", index=False)
summary.to_json(RESULT_ROOT / "summary_mean_std.json", orient="records", indent=2)

if not importance_df.empty:
    importance_summary = aggregate_mean_std(
        importance_df,
        ["feature_set", "selection_label", "regressor", "feature"],
        ["importance_mean"],
    ).sort_values("importance_mean_mean", ascending=False).reset_index(drop=True)
    importance_summary.to_csv(RESULT_ROOT / "permutation_importance_summary.csv", index=False)
    importance_df.to_csv(RESULT_ROOT / "permutation_importance_fold_level.csv", index=False)
else:
    importance_summary = pd.DataFrame()

print("Top grouped-CV configurations:")
display(summary.head(30))

# ---------------------------------------------------------------------
# 6) Plots
# ---------------------------------------------------------------------
top_summary = summary.head(min(25, len(summary))).copy()
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.barplot(data=top_summary, x="rmse_mean", y="feature_set", hue="selection_label", ax=axes[0], errorbar=None)
axes[0].set_title("Top feature sets by mean RMSE")
sns.barplot(data=top_summary, x="severity_balanced_accuracy_mean", y="feature_set", hue="selection_label", ax=axes[1], errorbar=None)
axes[1].set_title("Severity balanced accuracy from regression outputs")
for ax in axes:
    ax.legend(loc="best")
plt.tight_layout()
fig.savefig(PLOT_ROOT / "top_feature_sets.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(16, 6))
reg_view = summary.groupby("regressor", as_index=False)[["rmse_mean", "r2_mean"]].mean().sort_values("rmse_mean")
sns.barplot(data=reg_view, x="regressor", y="rmse_mean", ax=ax, errorbar=None)
ax.set_title("Mean RMSE by regressor across grouped-CV configurations")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
fig.savefig(PLOT_ROOT / "regressor_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

if not importance_summary.empty:
    top_importance = importance_summary.head(20).copy()
    fig, ax = plt.subplots(figsize=(14, max(6, 0.35 * len(top_importance))))
    sns.barplot(data=top_importance, x="importance_mean_mean", y="feature", hue="feature_set", ax=ax, errorbar=None)
    ax.set_title("Top aggregated permutation importances")
    plt.tight_layout()
    fig.savefig(PLOT_ROOT / "top_permutation_importances.png", dpi=180, bbox_inches="tight")
    plt.show()

# ---------------------------------------------------------------------
# 7) Save metadata + archive
# ---------------------------------------------------------------------
meta = {
    "repo": str(REPO),
    "inventory": str(INVENTORY),
    "group_column": GROUP_COL,
    "target": TARGET,
    "requested_folds": CV_FOLDS,
    "effective_folds": effective_folds,
    "strata_source": strat_source,
    "model_specs": MODEL_SPECS,
    "embedding_sources": embedding_sources,
    "top_k_values": TOP_K_VALUES,
    "max_pca_components": MAX_PCA_COMPONENTS,
    "severity_bins": SEVERITY_BINS,
    "frozen_backbone": True,
}
(RUN_ROOT / "run_config.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

archive_parent = Path("/kaggle/working") if Path("/kaggle/working").exists() else RUN_ROOT.parent
archive_base = archive_parent / f"grouped_cv_results_{RUN_TAG}"
archive_dir = archive_base.parent / archive_base.name
archive_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_ROOT, archive_dir / RUN_ROOT.name, dirs_exist_ok=True)
zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=archive_dir))
print(f"Created archive: {zip_path}")
display(FileLink(str(zip_path)))

## Representation Benchmark

This section prioritizes representation quality by caching deep features across mask and pooling variants, then benchmarking deep-only, handcrafted-only, and concatenated features under identical train/validation splits.

Priority order:
- representation choice first: deep-only vs handcrafted-only vs concatenated features
- mask strategy and pooling next: no mask, feature-map masking, masked input, crop-to-parcel bbox
- dimensionality reduction and regressors as secondary baselines


In [ ]:
from IPython.display import Image, display

REP_ROOT = ML_ROOT / "representation_experiments"
REP_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# PART-BASED EXECUTION CONTROLS
# -----------------------------
# Change PART_TAG each time you run a new chunk.
PART_TAG = "part_01"

# Use these to process only a subset of models/resolutions per part.
# Example: PART_MODEL_SLICE = (0, 2) processes MODEL_SPECS[0:2].
PART_MODEL_SLICE = (0, 2)
PART_RESOLUTIONS = ["sr"]

# If True, reuse already processed files and skip completed runs.
SKIP_EXISTING_FEATURES = True
SKIP_EXISTING_STAGE_SUMMARY = True

# Run toggles for this part.
RUN_STAGE1_CACHE = False
RUN_STAGE2_CACHE = False
RUN_STAGE1_BENCHMARK = False
RUN_STAGE2_BENCHMARK = False


def _subset_model_specs(specs, model_slice):
    if not model_slice:
        return list(specs)
    start, end = model_slice
    return list(specs[start:end])


REPRESENTATION_MODEL_SPECS = _subset_model_specs(MODEL_SPECS, PART_MODEL_SLICE)
REPRESENTATION_RESOLUTIONS = PART_RESOLUTIONS

PART_ROOT = REP_ROOT / "parts" / PART_TAG
PART_ROOT.mkdir(parents=True, exist_ok=True)

REPRESENTATION_CACHE_ROOT = PART_ROOT / "cache"
STAGE1_OUTPUT_DIR = PART_ROOT / "stage1_dr"
STAGE2_OUTPUT_DIR = PART_ROOT / "stage2_mask_pooling"

STAGE1_COMPONENT_GRID = [16, 32, 64, 128, 192, 256]
STAGE1_DR_METHODS = ["none", "pca", "incremental_pca", "truncated_svd", "pls"]
STAGE1_REGRESSORS = ["bayesian_ridge", "ridge", "random_forest", "extra_trees", "hist_gradient_boosting", "xgboost"]
STAGE2_MASK_POOLING_CONFIGS = [
    {"mask_fusion": "image_only", "pooling": "global_avg"},
    {"mask_fusion": "feature_mask_pool", "pooling": "global_avg"},
    {"mask_fusion": "feature_mask_pool", "pooling": "masked_avg"},
    {"mask_fusion": "feature_mask_pool", "pooling": "masked_mean_std"},
    {"mask_fusion": "masked_image", "pooling": "global_avg"},
    {"mask_fusion": "crop_mask_bbox", "pooling": "global_avg"},
]
STAGE2_REGRESSORS = STAGE1_REGRESSORS

print(f"Active part: {PART_TAG}")
print(f"Part root: {PART_ROOT}")
print(f"Models in this part: {len(REPRESENTATION_MODEL_SPECS)}")
print(f"Resolutions in this part: {REPRESENTATION_RESOLUTIONS}")


def representation_run_name(resolution, backbone_name, weight_name, mask_fusion, pooling):
    short_name = weight_name.split(".")[-1].lower()
    short_name = short_name.replace("sentinel2_", "").replace("weights", "")
    short_name = short_name.replace(":", "_").replace("/", "_")
    return f"{resolution}_{backbone_name.split(':')[-1]}_{short_name}_{mask_fusion}_{pooling}"


def _is_satlas_weight(weight_name):
    return "SATLAS" in weight_name.upper()


def _embedding_ready(out_dir):
    csv_path = out_dir / "backbone_embeddings.csv"
    return csv_path.exists() and csv_path.stat().st_size > 0


def run_representation_feature_extraction(cache_root, configs):
    planned = len(REPRESENTATION_RESOLUTIONS) * len(REPRESENTATION_MODEL_SPECS) * len(configs)
    print(f"Planned cached feature extraction runs: {planned}")

    ran = 0
    skipped = 0
    failed = 0

    for resolution in REPRESENTATION_RESOLUTIONS:
        for backbone, weight, _ in REPRESENTATION_MODEL_SPECS:
            for config in configs:
                run_name = representation_run_name(resolution, backbone, weight, config["mask_fusion"], config["pooling"])
                out_dir = cache_root / run_name

                if SKIP_EXISTING_FEATURES and _embedding_ready(out_dir):
                    skipped += 1
                    print(f"SKIP existing embeddings: {out_dir / 'backbone_embeddings.csv'}")
                    continue

                batch_size = "8" if _is_satlas_weight(weight) else "16"
                cmd = [
                    python_bin,
                    "scripts/train_yellowness_backbone_ml.py",
                    "--inventory", str(INVENTORY),
                    "--root-dir", str(DATASET_DIR),
                    "--resolution", resolution,
                    "--backbone", backbone,
                    "--torchgeo-weight", weight,
                    "--mask-fusion", config["mask_fusion"],
                    "--pooling", config["pooling"],
                    "--no-data-parallel",
                    "--batch-size", batch_size,
                    "--num-workers", "4",
                    "--extract-only",
                    "--feature-csv-name", "backbone_embeddings.csv",
                    "--output-dir", str(out_dir),
                ]
                ok = run_cmd(cmd, allow_failure=True)
                if ok and _embedding_ready(out_dir):
                    ran += 1
                else:
                    failed += 1
                    print(f"FAILED extraction run: {run_name}")

    print(f"Feature extraction finished | ran={ran}, skipped={skipped}, failed={failed}")


def run_representation_benchmark(cache_root, output_dir, dr_methods, component_grid, regressors):
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = output_dir / "representation_benchmark_summary.csv"

    if SKIP_EXISTING_STAGE_SUMMARY and summary_path.exists() and summary_path.stat().st_size > 0:
        print(f"SKIP existing benchmark summary: {summary_path}")
        return True

    cmd = [
        python_bin,
        "scripts/run_yellowness_representation_benchmark.py",
        "--inventory", str(INVENTORY),
        "--embedding-glob", str(cache_root / "*" / "backbone_embeddings.csv"),
        "--output-dir", str(output_dir),
        "--max-ml-dim", "300",
        "--component-grid", *[str(v) for v in component_grid],
        "--dr-methods", *dr_methods,
        "--regressors", *regressors,
    ]
    ok = run_cmd(cmd, allow_failure=True)
    if not ok:
        print(f"Benchmark command failed for output_dir={output_dir}")
        return False

    if summary_path.exists() and summary_path.stat().st_size > 0:
        print(f"Saved benchmark summary: {summary_path}")
        return True

    print(f"Benchmark finished but summary not found: {summary_path}")
    return False


if RUN_STAGE1_CACHE:
    run_representation_feature_extraction(
        REPRESENTATION_CACHE_ROOT / "stage1_dr",
        [{"mask_fusion": "feature_mask_pool", "pooling": "global_avg"}],
    )

if RUN_STAGE1_BENCHMARK:
    run_representation_benchmark(
        cache_root=REPRESENTATION_CACHE_ROOT / "stage1_dr",
        output_dir=STAGE1_OUTPUT_DIR,
        dr_methods=STAGE1_DR_METHODS,
        component_grid=STAGE1_COMPONENT_GRID,
        regressors=STAGE1_REGRESSORS,
    )

stage1_summary_path = STAGE1_OUTPUT_DIR / "representation_benchmark_summary.csv"
best_stage1_dr_method = "pca"
best_stage1_component = 64
if stage1_summary_path.exists() and stage1_summary_path.stat().st_size > 0:
    stage1_summary = pd.read_csv(stage1_summary_path)
    stage1_summary = stage1_summary.sort_values(["val_rmse", "val_r2", "val_mae"], ascending=[True, False, True]).reset_index(drop=True)
    best_stage1 = stage1_summary.iloc[0]
    best_stage1_dr_method = str(best_stage1["dr_method"])
    best_stage1_component = int(best_stage1["n_components"]) if not pd.isna(best_stage1["n_components"]) else 64
    display(stage1_summary.head(20))
    print(f"Best DR stage: {best_stage1_dr_method} | components={best_stage1_component}")
else:
    print(f"Stage 1 summary not found yet: {stage1_summary_path}")

if RUN_STAGE2_CACHE:
    run_representation_feature_extraction(REPRESENTATION_CACHE_ROOT / "stage2_mask_pooling", STAGE2_MASK_POOLING_CONFIGS)

if RUN_STAGE2_BENCHMARK:
    stage2_dr_methods = ["none"] if best_stage1_dr_method == "none" else ["none", best_stage1_dr_method]
    stage2_components = [best_stage1_component] if best_stage1_dr_method != "none" else [16]
    run_representation_benchmark(
        cache_root=REPRESENTATION_CACHE_ROOT / "stage2_mask_pooling",
        output_dir=STAGE2_OUTPUT_DIR,
        dr_methods=stage2_dr_methods,
        component_grid=stage2_components,
        regressors=STAGE2_REGRESSORS,
    )

stage2_summary_path = STAGE2_OUTPUT_DIR / "representation_benchmark_summary.csv"
if stage2_summary_path.exists() and stage2_summary_path.stat().st_size > 0:
    stage2_summary = pd.read_csv(stage2_summary_path)
    stage2_summary = stage2_summary.sort_values(["val_rmse", "val_r2", "val_mae"], ascending=[True, False, True]).reset_index(drop=True)
    display(stage2_summary.head(20))
else:
    print(f"Stage 2 summary not found yet: {stage2_summary_path}")

In [ ]:
import os
import shutil
from datetime import datetime
from pathlib import Path
from IPython.display import FileLink, Image, display

REPRESENTATION_BENCH_ROOT = ML_ROOT / "representation_benchmark"
REPRESENTATION_BENCH_ROOT.mkdir(parents=True, exist_ok=True)

# Keep False if you only want to visualize existing processed results.
RUN_REPRESENTATION_BENCHMARK = False
SKIP_EXISTING_MAIN_BENCHMARK = True

# Set True to include previously processed summaries from other parts/global runs in comparisons.
INCLUDE_PREVIOUS_SUMMARIES = True

# Set True to create a per-part zip after visualization/comparison.
CREATE_PART_ARCHIVE = True
INCLUDE_CACHE_IN_ARCHIVE = False

REP_COMPONENT_GRID = [16, 32, 64, 128, 192, 256]
REP_DR_METHODS = ["none", "pca", "incremental_pca", "truncated_svd", "pls"]
REP_REGRESSORS = [
    "bayesian_ridge",
    "ridge",
    "random_forest",
    "extra_trees",
    "hist_gradient_boosting",
    "xgboost",
    "lightgbm",
    "catboost",
]

# Use part variables created in Cell 10.
active_part_tag = globals().get("PART_TAG", "part_unknown")
active_part_root = globals().get("PART_ROOT", REP_ROOT / "parts" / active_part_tag)
active_part_root = Path(active_part_root)

cache_roots = []
if "REPRESENTATION_CACHE_ROOT" in globals():
    cache_roots.append(Path(REPRESENTATION_CACHE_ROOT))
if "REP_ROOT" in globals():
    cache_roots.extend([
        Path(REP_ROOT) / "cache",
        Path(REP_ROOT) / "cache" / "stage1_dr",
        Path(REP_ROOT) / "cache" / "stage2_mask_pooling",
    ])

seen = set()
cache_roots = [r for r in cache_roots if str(r) not in seen and not seen.add(str(r)) and r.exists()]

embedding_files = []
for root in cache_roots:
    embedding_files.extend(sorted(root.glob("**/backbone_embeddings.csv")))

print(f"Active part: {active_part_tag}")
print(f"Discovered {len(embedding_files)} embedding CSV files across {len(cache_roots)} cache roots.")
for p in embedding_files[:10]:
    print(" -", p)
if len(embedding_files) > 10:
    print(f"... and {len(embedding_files) - 10} more")

main_summary_path = REPRESENTATION_BENCH_ROOT / "representation_benchmark_summary.csv"
if RUN_REPRESENTATION_BENCHMARK:
    if SKIP_EXISTING_MAIN_BENCHMARK and main_summary_path.exists() and main_summary_path.stat().st_size > 0:
        print(f"SKIP existing benchmark summary: {main_summary_path}")
    elif not embedding_files:
        print("No backbone_embeddings.csv found. Run the feature extraction cell first.")
    else:
        common_root = Path(os.path.commonpath([str(p.parent) for p in embedding_files]))
        embedding_glob = str(common_root / "**" / "backbone_embeddings.csv")
        print(f"Using embedding glob: {embedding_glob}")
        cmd = [
            python_bin,
            "scripts/run_yellowness_representation_benchmark.py",
            "--inventory", str(INVENTORY),
            "--embedding-glob", embedding_glob,
            "--output-dir", str(REPRESENTATION_BENCH_ROOT),
            "--max-ml-dim", "300",
            "--component-grid", *[str(v) for v in REP_COMPONENT_GRID],
            "--dr-methods", *REP_DR_METHODS,
            "--regressors", *REP_REGRESSORS,
        ]
        ok = run_cmd(cmd, allow_failure=True)
        if not ok:
            print("Benchmark command failed, keeping existing outputs (if any).")

summary_paths = sorted(
    set(
        [
            *(active_part_root.glob("**/representation_benchmark_summary.csv") if active_part_root.exists() else []),
            *(ML_ROOT.glob("**/representation_benchmark_summary.csv") if INCLUDE_PREVIOUS_SUMMARIES else []),
            *(Path(REP_ROOT).glob("**/representation_benchmark_summary.csv") if INCLUDE_PREVIOUS_SUMMARIES and "REP_ROOT" in globals() else []),
            REPRESENTATION_BENCH_ROOT / "representation_benchmark_summary.csv",
        ]
    )
)
summary_paths = [p for p in summary_paths if p.exists() and p.stat().st_size > 0]

if summary_paths:
    print("Found benchmark summaries:")
    for p in summary_paths:
        print(" -", p)

    frames = []
    for p in summary_paths:
        try:
            df = pd.read_csv(p)
            if df.empty:
                continue
            df["summary_source"] = str(p)
            # Label rows by part for easier cross-part comparison.
            if "parts" in p.parts:
                part_idx = p.parts.index("parts")
                if part_idx + 1 < len(p.parts):
                    df["part_tag"] = p.parts[part_idx + 1]
                else:
                    df["part_tag"] = "unknown_part"
            else:
                df["part_tag"] = "legacy_or_global"
            frames.append(df)
        except Exception as exc:
            print(f"SKIP unreadable summary {p}: {exc}")

    if frames:
        representation_summary = pd.concat(frames, ignore_index=True)
        sort_cols = [c for c in ["val_rmse", "val_r2", "val_mae"] if c in representation_summary.columns]
        if sort_cols:
            ascending = [True if c != "val_r2" else False for c in sort_cols]
            representation_summary = representation_summary.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

        # Save merged comparison table for this notebook state.
        merged_path = active_part_root / f"merged_summary_{active_part_tag}.csv"
        merged_path.parent.mkdir(parents=True, exist_ok=True)
        representation_summary.to_csv(merged_path, index=False)
        print(f"Saved merged summary: {merged_path}")

        display(representation_summary.head(30))

        group_cols = [c for c in ["part_tag", "representation", "mask_fusion", "pooling"] if c in representation_summary.columns]
        metric_cols = [c for c in ["val_rmse", "val_r2"] if c in representation_summary.columns]
        if group_cols and metric_cols:
            grouped = (
                representation_summary.groupby(group_cols, as_index=False)[metric_cols]
                .mean()
                .sort_values(metric_cols, ascending=[True, False][: len(metric_cols)])
            )
            display(grouped.head(30))

        top = representation_summary.head(min(25, len(representation_summary))).copy()
        if {"dr_label", "val_rmse"}.issubset(top.columns):
            fig, ax = plt.subplots(figsize=(12, max(6, 0.32 * len(top))))
            hue_col = "regressor" if "regressor" in top.columns else None
            sns.barplot(data=top, x="val_rmse", y="dr_label", hue=hue_col, ax=ax, errorbar=None)
            ax.set_title("Top DR configurations (lower RMSE is better)")
            plt.tight_layout()
            plt.show()

        if {"mask_fusion", "pooling", "val_rmse"}.issubset(top.columns):
            fig, ax = plt.subplots(figsize=(14, max(6, 0.32 * len(top))))
            sns.barplot(data=top, x="val_rmse", y="mask_fusion", hue="pooling", ax=ax, errorbar=None)
            ax.set_title("Top mask/pooling configurations (lower RMSE is better)")
            plt.tight_layout()
            plt.show()

        representation_plots = [
            "predicted_vs_true_top_configs.png",
            "residuals_top_configs.png",
            "feature_dim_vs_performance.png",
            "representation_comparison.png",
            "mask_strategy_comparison.png",
            "pooling_comparison.png",
            "dr_method_comparison.png",
            "regressor_comparison.png",
        ]
        shown = set()
        for root in [active_part_root, REPRESENTATION_BENCH_ROOT, *{p.parent for p in summary_paths}]:
            for plot_name in representation_plots:
                plot_path = Path(root) / plot_name
                if plot_path.exists() and str(plot_path) not in shown:
                    print(plot_path)
                    display(Image(filename=str(plot_path)))
                    shown.add(str(plot_path))

        if CREATE_PART_ARCHIVE:
            archive_base = Path("/kaggle/working") / f"{active_part_tag}_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            archive_dir = archive_base.parent / archive_base.name
            archive_dir.mkdir(parents=True, exist_ok=True)

            # Always include part outputs and merged summary.
            items_to_collect = [
                active_part_root,
                PLOT_ROOT,
            ]
            if INCLUDE_CACHE_IN_ARCHIVE:
                items_to_collect.append(active_part_root / "cache")

            for item in items_to_collect:
                if not Path(item).exists():
                    continue
                target = archive_dir / Path(item).name
                if Path(item).is_dir():
                    shutil.copytree(item, target, dirs_exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(item, target)

            zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=archive_dir))
            print(f"Created part archive: {zip_path}")
            display(FileLink(str(zip_path)))
    else:
        print("No readable benchmark summaries found.")
else:
    print("No representation_benchmark_summary.csv found.")
    print("Set RUN_REPRESENTATION_BENCHMARK = True in this cell to generate a benchmark summary from discovered embeddings.")

## Deep Model Training

Run the end-to-end regressor training sweep for the same backbone families. These runs save checkpoints, run configs, and epoch histories that feed the loss-evolution plots at the end.


In [ ]:
# Run the deep-model sweep. This is slower than the classical feature pipeline, but it saves the training history needed for loss plots.

def run_deep_experiments(experiments, epochs=40, batch_size=8, num_workers=4):
    for spec in experiments:
        out_dir = DEEP_ROOT / spec["run_name"]
        cmd = [
            python_bin,
            "scripts/train_yellowness_regressor.py",
            "--inventory", str(INVENTORY),
            "--root-dir", str(DATASET_DIR),
            "--resolution", spec["resolution"],
            "--backbone", spec["backbone"],
            "--torchgeo-weight", spec["weight"],
            "--mask-fusion", "feature_mask_pool",
            "--freeze-backbone",
            "--unfreeze-epoch", "25",
            "--epochs", str(epochs),
            "--batch-size", str(batch_size),
            "--num-workers", str(num_workers),
            "--learning-rate", "1e-3",
            "--backbone-learning-rate", "1e-4",
            "--sample-patch-size", "224",
            "--center-crop-size", "224",
            "--rotate-augment",
            "--data-parallel",
            "--output-dir", str(out_dir),
        ]
        run_cmd(cmd)

# Uncomment one line at a time if you want to split the run into manageable chunks.
# run_deep_experiments(SR_EXPERIMENTS)
# run_deep_experiments(LR_EXPERIMENTS)


## Load Saved Results

Reload the saved classical-model summaries and deep-training histories before generating the comparison plots.


In [ ]:
def save_fig(fig, filename):
    path = PLOT_ROOT / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    return path


def load_ml_results():
    frames = []
    for path in sorted(ML_ROOT.glob("sr_*/summary.csv")) + sorted(ML_ROOT.glob("lr_*/summary.csv")):
        if not path.exists():
            continue
        df = pd.read_csv(path)
        df["run"] = path.parent.name
        df["resolution"] = "sr" if path.parent.name.startswith("sr_") else "lr"
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_deep_history():
    rows = []
    for path in sorted(DEEP_ROOT.glob("**/training_history.json")):
        run_dir = path.parent
        try:
            history = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        meta = {"run": run_dir.name}
        config_path = run_dir / "run_config.json"
        if config_path.exists():
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
                meta.update(
                    {
                        "resolution": config.get("resolution"),
                        "backbone": config.get("backbone"),
                        "torchgeo_weight": config.get("torchgeo_weight"),
                        "mask_fusion": config.get("mask_fusion"),
                    }
                )
            except Exception:
                pass
        for row in history:
            rows.append({**meta, **row})
    return pd.DataFrame(rows)


ml_results = load_ml_results()
deep_history = load_deep_history()

display(ml_results.head())
display(deep_history.head())

In [ ]:
def best_ml_rows(df):
    if df.empty:
        return df
    key_cols = ["resolution", "backbone", "regressor"]
    best_idx = df.groupby(key_cols)["val_rmse"].idxmin()
    return df.loc[best_idx].reset_index(drop=True)


def plot_sr_vs_lr_metrics(df):
    if df.empty:
        print("No ML results found yet.")
        return pd.DataFrame(), pd.DataFrame()

    best = best_ml_rows(df)
    agg = best.groupby(["resolution", "regressor"], as_index=False)[["train_rmse", "val_rmse", "train_mae", "val_mae", "train_r2", "val_r2"]].mean()

    figure_specs = [
        ("train_rmse", "val_rmse", "RMSE"),
        ("train_mae", "val_mae", "MAE"),
        ("train_r2", "val_r2", "R2"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(22, 6), sharey=False)
    for ax, (train_col, val_col, metric_name) in zip(axes, figure_specs):
        melted = agg.melt(
            id_vars=["resolution", "regressor"],
            value_vars=[train_col, val_col],
            var_name="split",
            value_name="value",
        )
        melted["split"] = melted["split"].map({train_col: "train", val_col: "validation"})
        melted["series"] = melted["resolution"].str.upper() + " / " + melted["split"]
        sns.barplot(data=melted, x="regressor", y="value", hue="series", ax=ax, errorbar=None)
        ax.set_title(f"{metric_name}: train vs validation")
        ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    saved = save_fig(fig, "sr_vs_lr_metrics.png")
    print(f"Saved: {saved}")
    plt.show()

    return best, agg


def plot_backbone_comparison(df):
    if df.empty:
        return
    best = best_ml_rows(df)
    agg = best.groupby(["resolution", "backbone"], as_index=False)["val_rmse"].mean()
    fig, ax = plt.subplots(figsize=(18, 6))
    sns.barplot(data=agg, x="backbone", y="val_rmse", hue="resolution", ax=ax, errorbar=None)
    ax.set_title("SR vs LR by backbone (best validation RMSE per run)")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    saved = save_fig(fig, "sr_vs_lr_backbones.png")
    print(f"Saved: {saved}")
    plt.show()


def plot_deep_loss_evolution(history_df, top_n=6):
    if history_df.empty:
        print("No deep training history found yet.")
        return

    hist = history_df.copy()
    hist["epoch"] = hist["epoch"].astype(int)
    ranked_runs = (
        hist.groupby(["run", "resolution", "backbone"], as_index=False)["val_loss"]
        .min()
        .sort_values("val_loss")
        .head(top_n)
    )
    selected = hist.merge(ranked_runs[["run"]], on="run", how="inner")

    fig, axes = plt.subplots(len(ranked_runs), 1, figsize=(14, max(4, 4 * len(ranked_runs))), sharex=True)
    if len(ranked_runs) == 1:
        axes = [axes]

    for ax, run_name in zip(axes, ranked_runs["run"].tolist()):
        run_df = selected[selected["run"] == run_name].sort_values("epoch")
        sns.lineplot(data=run_df, x="epoch", y="train_loss", ax=ax, label="train")
        sns.lineplot(data=run_df, x="epoch", y="val_loss", ax=ax, label="validation")
        meta = run_df.iloc[0]
        ax.set_title(f"{meta.get('resolution', '')} | {meta.get('backbone', '')} | {run_name}")
        ax.set_ylabel("Loss")

    plt.tight_layout()
    saved = save_fig(fig, "deep_loss_evolution.png")
    print(f"Saved: {saved}")
    plt.show()


best_ml_results, ml_aggregated = plot_sr_vs_lr_metrics(ml_results)
plot_backbone_comparison(ml_results)
plot_deep_loss_evolution(deep_history)

## Outputs And Saved Artifacts

Each run writes its own model artifacts, metric tables, and figure files.

Saved outputs include:
- `summary.csv` and `summary.json` for per-run metrics
- `model.pkl` for the fitted classical model and preprocessing stack
- `representation_benchmark_summary.csv` plus per-configuration predictions, metrics, and plots for the representation benchmark
- `best_model.pt`, `training_history.json`, and `run_config.json` for deep runs
- `backbone_embeddings.csv` and `backbone_embeddings.npz` for reusable embeddings
- saved plots under `outputs/kaggle_plots/`


In [ ]:
import shutil
from datetime import datetime
from IPython.display import FileLink, display

# Zip all experiment outputs into one archive under /kaggle/working
archive_base = Path("/kaggle/working") / f"yellowness_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
archive_dir = archive_base.parent / archive_base.name
archive_dir.mkdir(parents=True, exist_ok=True)

items_to_collect = [
    REPO / "outputs" / "yellowness_backbone_ml",
    REPO / "outputs" / "yellowness_regression",
    REPO / "outputs" / "kaggle_plots",
]

for item in items_to_collect:
    if item.exists():
        target = archive_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)
    else:
        print(f"Skipped missing output path: {item}")

zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=archive_dir))
print(f"Created archive: {zip_path}")
print("Download from the link below or from Kaggle Files pane (/kaggle/working).")
display(FileLink(str(zip_path)))

In [ ]:
# Standalone leakage-free grouped CV benchmark for spectral + frozen deep features
from pathlib import Path
from datetime import datetime
import json
import shutil
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import FileLink, display
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import BayesianRidge, RidgeCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

sns.set_theme(style="whitegrid", context="talk")

# ---------------------------------------------------------------------
# 1) Standalone configuration: change only this block for Kaggle
# ---------------------------------------------------------------------
RUN_ON_KAGGLE = Path("/kaggle/working").exists()
REPO_CANDIDATES = [
    Path.cwd(),
    Path("/kaggle/working/S2-super-resolution"),
    Path("/kaggle/working/s2-super-resolution"),
]
REPO = next((
    cand for cand in REPO_CANDIDATES
    if (cand / "pyproject.toml").exists() and (cand / "scripts").exists()
), Path.cwd())

# Kaggle adaptation notes:
if RUN_ON_KAGGLE:
    print("Kaggle detected.")
    print("If your repo is elsewhere, set REPO manually below.")
    print("If embeddings are under /kaggle/working/outputs/yellowness_backbone_ml, leave EMBEDDING_ROOTS as-is.")

# Change this manually on Kaggle only if auto-detection picks the wrong repo path.
# Example: REPO = Path("/kaggle/working/S2-super-resolution")

INVENTORY = Path("/kaggle/input/datasets/ahmedtrabelsi88/yellow-beet/observations_inventory.csv")
DATASET_DIR = Path("/kaggle/input/datasets/ahmedtrabelsi88/yellow-beet")
TARGET = "yellowness"
GROUP_COL = "id_plot"
ROW_COL = "row_index"
PYTHON_BIN = sys.executable

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = Path("/kaggle/working/outputs/outputs") / "standalone_grouped_cv" / f"run_{RUN_TAG}"
RESULT_ROOT = RUN_ROOT / "results"
PLOT_ROOT = RUN_ROOT / "plots"
EMBED_CACHE_ROOT = RUN_ROOT / "generated_embeddings"
for path in [RUN_ROOT, RESULT_ROOT, PLOT_ROOT, EMBED_CACHE_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# Minimal switch: set to "sr" to run super-resolved data, keep "lr" for low-resolution.
RUN_RESOLUTION = "sr"

# Exactly one model per family, as requested.
BACKBONE_SPECS = [
    {
        "family": "resnet",
        "backbone": "torchgeo:resnet50",
        "weight": "ResNet50_Weights.SENTINEL2_SI_MS_SATLAS",
        "name_suffix": "resnet50_satlas",
        "match_terms": ["resnet50", "satlas"],
    },
    {
        "family": "resnet152",
        "backbone": "torchgeo:resnet152",
        "weight": "ResNet152_Weights.SENTINEL2_SI_MS_SATLAS",
        "name_suffix": "resnet152_satlas",
        "match_terms": ["resnet152", "satlas"],
    },
    {
        "family": "swin",
        "backbone": "torchgeo:swin_v2_t",
        "weight": "Swin_V2_T_Weights.SENTINEL2_SI_MS_SATLAS",
        "name_suffix": "swin_t_satlas",
        "match_terms": ["swin", "satlas"],
    },
    {
        "family": "swin_b",
        "backbone": "torchgeo:swin_v2_b",
        "weight": "Swin_V2_B_Weights.SENTINEL2_SI_MS_SATLAS",
        "name_suffix": "swin_b_satlas",
        "match_terms": ["swin", "b", "satlas"],
    },
]
MODEL_SPECS = [
    {
        "family": spec["family"],
        "resolution": RUN_RESOLUTION,
        "backbone": spec["backbone"],
        "weight": spec["weight"],
        "run_name": f"{RUN_RESOLUTION}_{spec['name_suffix']}" ,
        "match_terms": [RUN_RESOLUTION, *spec["match_terms"]],
    }
    for spec in BACKBONE_SPECS
]

# Where to search for cached embeddings first.
EMBEDDING_ROOTS = [
    REPO / "outputs" / "yellowness_backbone_ml",
    Path("/kaggle/working") / "outputs" / "yellowness_backbone_ml",
]

# If no embeddings are found, automatically extract them with the backbone script.
AUTO_EXTRACT_EMBEDDINGS = True
EXTRACT_BATCH_SIZE = 16
EXTRACT_NUM_WORKERS = 4
EXTRACT_MASK_FUSION = "image_only"
EXTRACT_POOLING = "masked_mean_std"
EXTRACT_SHAPEFILE_PATH = REPO / "data" / "shapefiles" / "2020_SEPIM.shp"
EXTRACT_MASK_BUFFER_M = -10.0

CV_FOLDS = 5
RANDOM_STATE = 42
CORR_THRESHOLD = 0.98
TOP_K_VALUES = [30, 40, 50]
MAX_PCA_COMPONENTS = 50
PERMUTATION_REPEATS = 8

SEVERITY_BINS = [-np.inf, 10, 60, np.inf]
SEVERITY_LABELS = ["low", "medium", "high"]

REGRESSOR_BUILDERS = {
    "bayesian_ridge": lambda: Pipeline([
        ("scale", StandardScaler()),
        ("model", BayesianRidge()),
    ]),
    "ridge": lambda: Pipeline([
        ("scale", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-3, 3, 13))),
    ]),
    "random_forest": lambda: RandomForestRegressor(
        n_estimators=220,
        min_samples_leaf=2,
        max_features=0.7,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "extra_trees": lambda: ExtraTreesRegressor(
        n_estimators=240,
        min_samples_leaf=2,
        max_features=0.7,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "hist_gradient_boosting": lambda: HistGradientBoostingRegressor(
        learning_rate=0.04,
        max_depth=6,
        max_iter=220,
        min_samples_leaf=8,
        l2_regularization=0.05,
        random_state=RANDOM_STATE,
    ),
    "svr": lambda: Pipeline([
        ("scale", StandardScaler()),
        ("model", SVR(C=10.0, epsilon=0.05, gamma="scale")),
    ]),
}

OPTIONAL_REGRESSOR_BUILDERS = {
    "xgboost": ("xgboost", "XGBRegressor"),
}

TREE_REGRESSORS = {"random_forest", "extra_trees", "xgboost"}

# ---------------------------------------------------------------------
# 2) Helpers
# ---------------------------------------------------------------------
def run_cmd(cmd, cwd=None, allow_failure=False):
    print("RUN:", " ".join(map(str, cmd)))
    try:
        subprocess.run([str(x) for x in cmd], check=True, cwd=str(cwd) if cwd else None)
        return True
    except subprocess.CalledProcessError as exc:
        print(f"FAILED: {exc}")
        if not allow_failure:
            raise
        return False


def metric_dict(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def severity_from_values(values):
    clipped = np.clip(np.asarray(values, dtype=float), 0.0, 100.0)
    return pd.cut(
        clipped,
        bins=SEVERITY_BINS,
        labels=SEVERITY_LABELS,
        include_lowest=True,
        right=True,
    ).astype(str)


def severity_metric_dict(y_true, y_pred):
    sev_true = severity_from_values(y_true)
    sev_pred = severity_from_values(y_pred)
    return {
        "severity_accuracy": float(accuracy_score(sev_true, sev_pred)),
        "severity_balanced_accuracy": float(balanced_accuracy_score(sev_true, sev_pred)),
        "severity_f1_macro": float(f1_score(sev_true, sev_pred, average="macro", zero_division=0)),
    }


def drop_high_corr(df_in, columns, threshold=0.98):
    if len(columns) <= 1:
        return list(columns)
    corr_mat = df_in[columns].corr(numeric_only=True).abs()
    upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return [col for col in columns if col not in to_drop]


def discover_embedding_csv(search_roots, spec):
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        candidates.extend(root.glob("**/*embeddings.csv"))
        candidates.extend(root.glob("**/backbone_embeddings.csv"))

    scored = []
    for path in candidates:
        text = str(path).lower()
        score = 0
        if spec["run_name"].lower() in text:
            score += 10
        score += sum(term in text for term in spec["match_terms"])
        if score > 0:
            scored.append((score, len(text), path))

    if not scored:
        return None

    scored.sort(key=lambda item: (-item[0], item[1], str(item[2])))
    return scored[0][2]


def extract_embedding_csv(spec):
    out_dir = EMBED_CACHE_ROOT / spec["run_name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / "backbone_embeddings.csv"
    if csv_path.exists() and csv_path.stat().st_size > 0:
        return csv_path

    cmd = [
        PYTHON_BIN,
        "scripts/train_yellowness_backbone_ml.py",
        "--inventory", str(INVENTORY),
        "--root-dir", str(DATASET_DIR),
        "--resolution", spec["resolution"],
        "--backbone", spec["backbone"],
        "--torchgeo-weight", spec["weight"],
        "--mask-fusion", EXTRACT_MASK_FUSION,
        "--pooling", EXTRACT_POOLING,
        "--shapefile-path", str(EXTRACT_SHAPEFILE_PATH),
        "--mask-buffer-m", str(EXTRACT_MASK_BUFFER_M),
        "--no-data-parallel",
        "--batch-size", str(EXTRACT_BATCH_SIZE),
        "--num-workers", str(EXTRACT_NUM_WORKERS),
        "--log-feature-shapes",
        "--extract-only",
        "--feature-csv-name", "backbone_embeddings.csv",
        "--output-dir", str(out_dir),
    ]
    ok = run_cmd(cmd, cwd=REPO, allow_failure=True)
    if ok and csv_path.exists() and csv_path.stat().st_size > 0:
        return csv_path
    raise FileNotFoundError(f"Embedding extraction failed for {spec['run_name']}: {csv_path}")


def load_embedding_table(spec):
    csv_path = discover_embedding_csv(EMBEDDING_ROOTS, spec)
    if csv_path is None and AUTO_EXTRACT_EMBEDDINGS:
        csv_path = extract_embedding_csv(spec)

    if csv_path is None:
        roots_text = "\n".join(str(p) for p in EMBEDDING_ROOTS)
        raise FileNotFoundError(
            f"No embedding CSV found for {spec['family']} ({spec['run_name']}). Searched:\n{roots_text}"
        )

    emb_df = pd.read_csv(csv_path)
    feat_cols = [col for col in emb_df.columns if col.startswith("feat_")]
    if ROW_COL not in emb_df.columns or not feat_cols:
        raise ValueError(f"Embedding file missing {ROW_COL} or feat_* columns: {csv_path}")

    emb_df = emb_df.dropna(subset=[ROW_COL]).copy()
    emb_df[ROW_COL] = emb_df[ROW_COL].astype(int)
    if emb_df[ROW_COL].duplicated().any():
        emb_df = emb_df.groupby(ROW_COL, as_index=False)[feat_cols].mean()
    else:
        emb_df = emb_df[[ROW_COL, *feat_cols]].copy()

    renamed = {col: f"{spec['family']}__{col}" for col in feat_cols}
    emb_df = emb_df.rename(columns=renamed)
    return csv_path, emb_df, list(renamed.values())


def resolve_group_strata(df, requested_folds):
    group_target = df.groupby(GROUP_COL)[TARGET].mean()
    if group_target.empty:
        raise RuntimeError("No parcel groups found after merging embeddings.")

    # Quantile bins at parcel level to preserve target distribution during grouped CV.
    for q in range(min(5, int(group_target.nunique())), 1, -1):
        try:
            bins = pd.qcut(group_target.rank(method="first"), q=q, labels=False, duplicates="drop")
        except Exception:
            continue
        bins = pd.Series(bins, index=group_target.index).astype(int)
        counts = bins.value_counts()
        folds = min(requested_folds, int(counts.min())) if not counts.empty else 0
        if folds >= 2:
            return bins, folds, f"quantile_{len(counts)}"

    severity_bins = pd.cut(
        group_target,
        bins=SEVERITY_BINS,
        labels=False,
        include_lowest=True,
        right=True,
    )
    severity_bins = pd.Series(severity_bins, index=group_target.index).fillna(0).astype(int)
    counts = severity_bins.value_counts()
    folds = min(requested_folds, int(counts.min())) if not counts.empty else 0
    if folds >= 2:
        return severity_bins, folds, "severity_fallback"

    raise RuntimeError("Unable to build at least 2 stratified parcel folds from current data.")


def build_optional_regressor(name):
    module_name, class_name = OPTIONAL_REGRESSOR_BUILDERS[name]
    try:
        module = __import__(module_name, fromlist=[class_name])
        cls = getattr(module, class_name)
    except Exception:
        return None

    if name == "xgboost":
        return cls(
            n_estimators=180,
            max_depth=5,
            learning_rate=0.04,
            subsample=0.9,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=4,
        )
    if name == "lightgbm":
        return cls(
            n_estimators=220,
            learning_rate=0.04,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.8,
            objective="regression",
            random_state=RANDOM_STATE,
        )
    return None


def build_regressors():
    models = {name: builder() for name, builder in REGRESSOR_BUILDERS.items()}
    for name in OPTIONAL_REGRESSOR_BUILDERS:
        model = build_optional_regressor(name)
        if model is not None:
            models[name] = model
        else:
            warnings.warn(f"Optional regressor unavailable and will be skipped: {name}")
    return models


def prepare_feature_block(train_df, val_df, feature_cols, y_train, selection_mode, selection_value=None):
    cols = [col for col in feature_cols if col in train_df.columns]
    if not cols:
        return None

    x_train_df = train_df[cols].copy()
    x_val_df = val_df[cols].copy()

    train_medians = x_train_df.median(numeric_only=True)
    x_train_df = x_train_df.replace([np.inf, -np.inf], np.nan).fillna(train_medians)
    x_val_df = x_val_df.replace([np.inf, -np.inf], np.nan).fillna(train_medians)

    valid_cols = [col for col in x_train_df.columns if x_train_df[col].nunique(dropna=True) > 1]
    if not valid_cols:
        return None

    x_train_df = x_train_df[valid_cols]
    x_val_df = x_val_df[valid_cols]
    decor_cols = drop_high_corr(x_train_df, list(x_train_df.columns), threshold=CORR_THRESHOLD)
    if not decor_cols:
        return None

    x_train_df = x_train_df[decor_cols]
    x_val_df = x_val_df[decor_cols]

    if selection_mode == "mi":
        top_k = min(int(selection_value), x_train_df.shape[1])
        if top_k < 1:
            return None
        mi_scores = mutual_info_regression(x_train_df, y_train, random_state=RANDOM_STATE)
        mi_series = pd.Series(mi_scores, index=x_train_df.columns).sort_values(ascending=False)
        selected_cols = mi_series.head(top_k).index.tolist()
        return {
            "X_train": x_train_df[selected_cols].to_numpy(dtype=np.float32),
            "X_val": x_val_df[selected_cols].to_numpy(dtype=np.float32),
            "feature_names": selected_cols,
            "selection_label": f"mi_top_{top_k:02d}",
            "decorrelated_dim": int(x_train_df.shape[1]),
            "final_dim": int(len(selected_cols)),
        }

    if selection_mode == "pca95":
        scaler = StandardScaler()
        x_train_sc = scaler.fit_transform(x_train_df)
        x_val_sc = scaler.transform(x_val_df)

        full_pca = PCA(random_state=RANDOM_STATE)
        full_pca.fit(x_train_sc)
        cumvar = np.cumsum(full_pca.explained_variance_ratio_)
        needed = int(np.searchsorted(cumvar, 0.95) + 1)
        n_components = max(1, min(needed, MAX_PCA_COMPONENTS, x_train_sc.shape[0] - 1, x_train_sc.shape[1]))

        reducer = PCA(n_components=n_components, random_state=RANDOM_STATE)
        x_train_pca = reducer.fit_transform(x_train_sc)
        x_val_pca = reducer.transform(x_val_sc)
        feature_names = [f"pc_{idx + 1:02d}" for idx in range(x_train_pca.shape[1])]
        return {
            "X_train": x_train_pca.astype(np.float32),
            "X_val": x_val_pca.astype(np.float32),
            "feature_names": feature_names,
            "selection_label": f"pca95_cap_{x_train_pca.shape[1]:02d}",
            "decorrelated_dim": int(x_train_df.shape[1]),
            "final_dim": int(x_train_pca.shape[1]),
        }

    raise ValueError(f"Unsupported selection mode: {selection_mode}")


def aggregate_mean_std(frame, group_cols, metric_cols):
    grouped = frame.groupby(group_cols, dropna=False)[metric_cols].agg(["mean", "std"]).reset_index()
    grouped.columns = [
        "_".join([str(part) for part in col if part]).rstrip("_")
        for col in grouped.columns.to_flat_index()
    ]
    for metric in metric_cols:
        grouped[f"{metric}_mean_std"] = grouped.apply(
            lambda row: f"{row[f'{metric}_mean']:.4f} +/- {0.0 if pd.isna(row[f'{metric}_std']) else row[f'{metric}_std']:.4f}",
            axis=1,
        )
    return grouped


# ---------------------------------------------------------------------
# 3) Load inventory + cached or newly extracted embeddings
# ---------------------------------------------------------------------
if not INVENTORY.exists():
    raise FileNotFoundError(f"Inventory not found: {INVENTORY}")
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

inventory_df = pd.read_csv(INVENTORY).reset_index(drop=True)
inventory_df[ROW_COL] = inventory_df.index.astype(int)

excluded_exact = {TARGET, GROUP_COL, ROW_COL, "year", "parcel_index", "source_file", "source_inventory"}
spectral_prefixes = ("lr_", "sr_", "calc_lr_", "calc_sr_")
meta_features = [col for col in ["month", "day_of_year", "s2_date_difference_days"] if col in inventory_df.columns]
spectral_features = [
    col
    for col in inventory_df.columns
    if pd.api.types.is_numeric_dtype(inventory_df[col])
    and col not in excluded_exact
    and inventory_df[col].nunique(dropna=True) > 1
    and (col.startswith(spectral_prefixes) or col in meta_features)
]

merged_df = inventory_df[[ROW_COL, GROUP_COL, TARGET, *sorted(set(spectral_features))]].copy()
deep_feature_sets = {}
embedding_sources = {}

for spec in MODEL_SPECS:
    csv_path, emb_df, deep_cols = load_embedding_table(spec)
    embedding_sources[spec["family"]] = str(csv_path)
    deep_feature_sets[spec["family"]] = deep_cols
    merged_df = merged_df.merge(emb_df, on=ROW_COL, how="inner")

if merged_df.empty:
    raise RuntimeError("No rows remain after aligning inventory with the embedding tables.")
if merged_df[GROUP_COL].nunique() < 2:
    raise RuntimeError("Need at least two parcels after alignment to run grouped CV.")

print(f"Aligned rows: {len(merged_df)}")
print(f"Aligned parcels: {merged_df[GROUP_COL].nunique()}")
print("Embedding sources:")
for family, path in embedding_sources.items():
    print(f" - {family}: {path}")
print(f"Active resolution mode: {RUN_RESOLUTION}")

feature_sets = {
    "spectral_only": spectral_features,
    "resnet_only": deep_feature_sets["resnet"],
    "swin_only": deep_feature_sets["swin"],
    "vit_only": deep_feature_sets["vit"],
    "resnet_plus_spectral": deep_feature_sets["resnet"] + spectral_features,
    "swin_plus_spectral": deep_feature_sets["swin"] + spectral_features,
    "vit_plus_spectral": deep_feature_sets["vit"] + spectral_features,
}

group_strata, effective_folds, strat_source = resolve_group_strata(merged_df, CV_FOLDS)
row_strata = merged_df[GROUP_COL].map(group_strata).astype(int)
print(f"Using {effective_folds} grouped folds with strata source: {strat_source}")
print(group_strata.value_counts().sort_index().to_string())

cv = StratifiedGroupKFold(n_splits=effective_folds, shuffle=True, random_state=RANDOM_STATE)
regressors = build_regressors()
selection_configs = [("mi", top_k) for top_k in TOP_K_VALUES] + [("pca95", MAX_PCA_COMPONENTS)]

all_rows = []
prediction_rows = []
importance_rows = []

# ---------------------------------------------------------------------
# 4) Leakage-free grouped CV
# ---------------------------------------------------------------------
for fold_id, (train_idx, val_idx) in enumerate(cv.split(merged_df, row_strata, groups=merged_df[GROUP_COL]), start=1):
    train_df = merged_df.iloc[train_idx].copy()
    val_df = merged_df.iloc[val_idx].copy()

    train_groups = set(train_df[GROUP_COL].astype(str))
    val_groups = set(val_df[GROUP_COL].astype(str))
    if train_groups & val_groups:
        raise RuntimeError("Grouped CV leakage detected: at least one parcel appears in both train and validation.")

    y_train = train_df[TARGET].to_numpy(dtype=np.float32)
    y_val = val_df[TARGET].to_numpy(dtype=np.float32)

    print(
        f"\nFold {fold_id}/{effective_folds} | train rows={len(train_df)} | val rows={len(val_df)} | "
        f"train parcels={train_df[GROUP_COL].nunique()} | val parcels={val_df[GROUP_COL].nunique()}"
    )

    for feature_set_name, feature_cols in feature_sets.items():
        for selection_mode, selection_value in selection_configs:
            prepared = prepare_feature_block(train_df, val_df, feature_cols, y_train, selection_mode, selection_value)
            if prepared is None or prepared["final_dim"] < 1:
                continue

            x_train_np = np.asarray(prepared["X_train"], dtype=np.float32)
            x_val_np = np.asarray(prepared["X_val"], dtype=np.float32)
            if x_train_np.ndim == 1:
                x_train_np = x_train_np.reshape(-1, 1)
            if x_val_np.ndim == 1:
                x_val_np = x_val_np.reshape(-1, 1)
            feature_names = prepared["feature_names"]
            if x_train_np.shape[1] != len(feature_names) or x_val_np.shape[1] != len(feature_names):
                raise RuntimeError("Feature matrix shape does not match selected feature order.")

            for reg_name, regressor in regressors.items():
                model = clone(regressor)
                fit_kwargs = {}
                if reg_name == "xgboost" and not isinstance(model, Pipeline):
                    fit_kwargs = {"eval_set": [(x_val_np, y_val)], "verbose": False}

                try:
                    model.fit(x_train_np, y_train, **fit_kwargs)
                    pred_val = np.asarray(model.predict(x_val_np)).reshape(-1)
                except Exception as exc:
                    warnings.warn(
                        f"Skipping {feature_set_name} | {prepared['selection_label']} | {reg_name} on fold {fold_id}: {exc}"
                    )
                    continue

                reg_metrics = metric_dict(y_val, pred_val)
                sev_metrics = severity_metric_dict(y_val, pred_val)

                all_rows.append({
                    "fold": fold_id,
                    "resolution": RUN_RESOLUTION,
                    "feature_set": feature_set_name,
                    "selection_mode": selection_mode,
                    "selection_value": selection_value,
                    "selection_label": prepared["selection_label"],
                    "regressor": reg_name,
                    "train_rows": int(len(train_df)),
                    "val_rows": int(len(val_df)),
                    "train_parcels": int(train_df[GROUP_COL].nunique()),
                    "val_parcels": int(val_df[GROUP_COL].nunique()),
                    "decorrelated_dim": prepared["decorrelated_dim"],
                    "feature_dim": prepared["final_dim"],
                    **reg_metrics,
                    **sev_metrics,
                })

                pred_frame = pd.DataFrame({
                    ROW_COL: val_df[ROW_COL].to_numpy(),
                    GROUP_COL: val_df[GROUP_COL].to_numpy(),
                    "fold": fold_id,
                    "resolution": RUN_RESOLUTION,
                    "feature_set": feature_set_name,
                    "selection_label": prepared["selection_label"],
                    "regressor": reg_name,
                    "y_true": y_val,
                    "y_pred": pred_val,
                    "severity_true": severity_from_values(y_val),
                    "severity_pred": severity_from_values(pred_val),
                })
                prediction_rows.append(pred_frame)

                if reg_name in TREE_REGRESSORS and len(feature_names) <= 50:
                    try:
                        perm = permutation_importance(
                            model,
                            x_val_np,
                            y_val,
                            n_repeats=PERMUTATION_REPEATS,
                            random_state=RANDOM_STATE,
                            scoring="neg_root_mean_squared_error",
                            n_jobs=1,
                        )
                        for feat_name, mean_imp, std_imp in zip(
                            feature_names, perm.importances_mean, perm.importances_std
                        ):
                            importance_rows.append({
                                "fold": fold_id,
                                "feature_set": feature_set_name,
                                "selection_label": prepared["selection_label"],
                                "regressor": reg_name,
                                "feature": feat_name,
                                "importance_mean": float(mean_imp),
                                "importance_std": float(std_imp),
                            })
                    except Exception as exc:
                        warnings.warn(
                            f"Permutation importance failed for {feature_set_name} | {prepared['selection_label']} | {reg_name} on fold {fold_id}: {exc}"
                        )

# ---------------------------------------------------------------------
# 5) Save results
# ---------------------------------------------------------------------
fold_results = pd.DataFrame(all_rows)
if fold_results.empty:
    raise RuntimeError("No successful model fits were produced by the grouped CV benchmark.")

predictions_df = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
importance_df = pd.DataFrame(importance_rows)

fold_results = fold_results.sort_values(["rmse", "r2", "mae"], ascending=[True, False, True]).reset_index(drop=True)
fold_results.to_csv(RESULT_ROOT / "fold_metrics.csv", index=False)
fold_results.to_json(RESULT_ROOT / "fold_metrics.json", orient="records", indent=2)

if not predictions_df.empty:
    predictions_df.to_csv(RESULT_ROOT / "fold_predictions.csv", index=False)

summary = aggregate_mean_std(
    fold_results,
    ["resolution", "feature_set", "selection_label", "regressor"],
    ["rmse", "mae", "r2", "severity_accuracy", "severity_balanced_accuracy", "severity_f1_macro"],
)
summary = summary.sort_values(["rmse_mean", "r2_mean", "mae_mean"], ascending=[True, False, True]).reset_index(drop=True)
summary.to_csv(RESULT_ROOT / "summary_mean_std.csv", index=False)
summary.to_json(RESULT_ROOT / "summary_mean_std.json", orient="records", indent=2)

if not importance_df.empty:
    importance_summary = aggregate_mean_std(
        importance_df,
        ["feature_set", "selection_label", "regressor", "feature"],
        ["importance_mean"],
    ).sort_values("importance_mean_mean", ascending=False).reset_index(drop=True)
    importance_summary.to_csv(RESULT_ROOT / "permutation_importance_summary.csv", index=False)
    importance_df.to_csv(RESULT_ROOT / "permutation_importance_fold_level.csv", index=False)
else:
    importance_summary = pd.DataFrame()

print("Top grouped-CV configurations:")
display(summary.head(30))

# ---------------------------------------------------------------------
# 6) Plots
# ---------------------------------------------------------------------
top_summary = summary.head(min(25, len(summary))).copy()
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.barplot(data=top_summary, x="rmse_mean", y="feature_set", hue="selection_label", ax=axes[0], errorbar=None)
axes[0].set_title("Top feature sets by mean RMSE")
sns.barplot(
    data=top_summary,
    x="severity_balanced_accuracy_mean",
    y="feature_set",
    hue="selection_label",
    ax=axes[1],
    errorbar=None,
 )
axes[1].set_title("Severity balanced accuracy from regression outputs")
for ax in axes:
    ax.legend(loc="best")
plt.tight_layout()
fig.savefig(PLOT_ROOT / "top_feature_sets.png", dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(16, 6))
reg_view = summary.groupby("regressor", as_index=False)[["rmse_mean", "r2_mean"]].mean().sort_values("rmse_mean")
sns.barplot(data=reg_view, x="regressor", y="rmse_mean", ax=ax, errorbar=None)
ax.set_title("Mean RMSE by regressor across grouped-CV configurations")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
fig.savefig(PLOT_ROOT / "regressor_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

# Prediction vs observation plots for top configurations (saved for SR/LR effect analysis).
top_cfg_cols = ["resolution", "feature_set", "selection_label", "regressor"]
top_cfg = summary[top_cfg_cols].head(min(9, len(summary))).drop_duplicates()
if not predictions_df.empty and not top_cfg.empty:
    pred_top = predictions_df.merge(top_cfg, on=top_cfg_cols, how="inner")
    if not pred_top.empty:
        pred_top = pred_top.copy()
        pred_top["config"] = pred_top.apply(
            lambda r: f"{r['regressor']} | {r['feature_set']} | {r['selection_label']}", axis=1
        )
        configs = pred_top["config"].drop_duplicates().tolist()
        n_cfg = len(configs)
        n_cols = 3
        n_rows = int(np.ceil(n_cfg / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows), squeeze=False)
        for idx, cfg in enumerate(configs):
            ax = axes[idx // n_cols][idx % n_cols]
            sub = pred_top[pred_top["config"] == cfg]
            ax.scatter(sub["y_true"], sub["y_pred"], alpha=0.6, s=20)
            mn = float(min(sub["y_true"].min(), sub["y_pred"].min()))
            mx = float(max(sub["y_true"].max(), sub["y_pred"].max()))
            ax.plot([mn, mx], [mn, mx], linestyle="--", linewidth=1.2, color="black")
            ax.set_title(cfg)
            ax.set_xlabel("Observation")
            ax.set_ylabel("Prediction")
        for idx in range(n_cfg, n_rows * n_cols):
            axes[idx // n_cols][idx % n_cols].axis("off")
        plt.tight_layout()
        fig.savefig(PLOT_ROOT / f"pred_vs_obs_top_configs_{RUN_RESOLUTION}.png", dpi=180, bbox_inches="tight")
        plt.show()

if not importance_summary.empty:
    top_importance = importance_summary.head(20).copy()
    fig, ax = plt.subplots(figsize=(14, max(6, 0.35 * len(top_importance))))
    sns.barplot(
        data=top_importance,
        x="importance_mean_mean",
        y="feature",
        hue="feature_set",
        ax=ax,
        errorbar=None,
    )
    ax.set_title("Top aggregated permutation importances")
    plt.tight_layout()
    fig.savefig(PLOT_ROOT / "top_permutation_importances.png", dpi=180, bbox_inches="tight")
    plt.show()

# ---------------------------------------------------------------------
# 7) Save metadata + archive
# ---------------------------------------------------------------------
meta = {
    "repo": str(REPO),
    "inventory": str(INVENTORY),
    "dataset_dir": str(DATASET_DIR),
    "group_column": GROUP_COL,
    "target": TARGET,
    "run_resolution": RUN_RESOLUTION,
    "requested_folds": CV_FOLDS,
    "effective_folds": effective_folds,
    "strata_source": strat_source,
    "model_specs": MODEL_SPECS,
    "embedding_sources": embedding_sources,
    "auto_extract_embeddings": AUTO_EXTRACT_EMBEDDINGS,
    "top_k_values": TOP_K_VALUES,
    "max_pca_components": MAX_PCA_COMPONENTS,
    "severity_bins": SEVERITY_BINS,
    "frozen_backbone": True,
}
(RUN_ROOT / "run_config.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

archive_parent = Path("/kaggle/working") if RUN_ON_KAGGLE else RUN_ROOT.parent
archive_base = archive_parent / f"grouped_cv_results_{RUN_TAG}"
archive_dir = archive_base.parent / archive_base.name
archive_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_ROOT, archive_dir / RUN_ROOT.name, dirs_exist_ok=True)
zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=archive_dir))
print(f"Created archive: {zip_path}")
display(FileLink(str(zip_path)))

FileNotFoundError: Inventory not found: c:\Users\ahtrabelsi\Desktop\stage\s2-super-resolution - preso\notebooks\yellowness_dataset\observations_inventory.csv

In [ ]:
# SR vs LR comparison from saved standalone grouped-CV runs
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")

run_roots = [
    Path("/kaggle/working/outputs/outputs/standalone_grouped_cv"),
    Path("/kaggle/working/outputs/standalone_grouped_cv"),
    REPO / "outputs" / "outputs" / "standalone_grouped_cv",
    REPO / "outputs" / "standalone_grouped_cv",
]

candidate_runs = []
for root in run_roots:
    if root.exists():
        candidate_runs.extend(sorted(root.glob("run_*/results/summary_mean_std.csv")))

if not candidate_runs:
    print("No standalone grouped-CV summaries found yet. Run Cell 20 for LR and SR first.")
else:
    records = []
    pred_frames = []
    for summary_path in candidate_runs:
        run_dir = summary_path.parents[1]
        config_path = run_dir / "run_config.json"
        pred_path = run_dir / "results" / "fold_predictions.csv"
        if not config_path.exists():
            continue

        cfg = pd.read_json(config_path, typ="series")
        run_resolution = str(cfg.get("run_resolution", "")).lower()
        if run_resolution not in {"sr", "lr"}:
            continue

        try:
            s = pd.read_csv(summary_path)
        except Exception:
            continue

        s["resolution"] = run_resolution
        s["run_dir"] = str(run_dir)
        records.append(s)

        if pred_path.exists():
            p = pd.read_csv(pred_path)
            p["resolution"] = run_resolution
            p["run_dir"] = str(run_dir)
            pred_frames.append(p)

    if not records:
        print("No valid run summaries with run_resolution in run_config.json were found.")
    else:
        summary_all = pd.concat(records, ignore_index=True)
        summary_all = summary_all.sort_values(["rmse_mean", "r2_mean"], ascending=[True, False]).reset_index(drop=True)

        cmp_out = Path("/kaggle/working") if Path("/kaggle/working").exists() else summary_all.iloc[0]["run_dir"]
        cmp_out = Path(cmp_out) / "sr_lr_comparison"
        cmp_out.mkdir(parents=True, exist_ok=True)

        summary_all.to_csv(cmp_out / "summary_all_runs.csv", index=False)

        key_cols = ["feature_set", "selection_label", "regressor"]
        metric_cols = ["rmse_mean", "mae_mean", "r2_mean", "severity_balanced_accuracy_mean"]

        best_per_res = summary_all.groupby(["resolution", *key_cols], as_index=False)[metric_cols].mean()
        sr = best_per_res[best_per_res["resolution"] == "sr"].copy()
        lr = best_per_res[best_per_res["resolution"] == "lr"].copy()

        if sr.empty or lr.empty:
            print("Need both SR and LR runs to compare. Currently available resolutions:", best_per_res["resolution"].unique())
        else:
            merged = sr.merge(lr, on=key_cols, suffixes=("_sr", "_lr"))
            merged["delta_rmse_sr_minus_lr"] = merged["rmse_mean_sr"] - merged["rmse_mean_lr"]
            merged["delta_mae_sr_minus_lr"] = merged["mae_mean_sr"] - merged["mae_mean_lr"]
            merged["delta_r2_sr_minus_lr"] = merged["r2_mean_sr"] - merged["r2_mean_lr"]
            merged["delta_sev_bal_acc_sr_minus_lr"] = (
                merged["severity_balanced_accuracy_mean_sr"] - merged["severity_balanced_accuracy_mean_lr"]
            )

            merged = merged.sort_values("delta_rmse_sr_minus_lr").reset_index(drop=True)
            merged.to_csv(cmp_out / "sr_vs_lr_delta_table.csv", index=False)

            print("Top improvements from SR (negative delta_rmse_sr_minus_lr is better):")
            display(merged.head(20))

            top_n = min(20, len(merged))
            top_delta = merged.head(top_n).copy()
            top_delta["config"] = top_delta.apply(
                lambda r: f"{r['regressor']} | {r['feature_set']} | {r['selection_label']}", axis=1
            )

            fig, axes = plt.subplots(1, 2, figsize=(20, max(6, 0.35 * top_n)))
            sns.barplot(data=top_delta, x="delta_rmse_sr_minus_lr", y="config", ax=axes[0], errorbar=None)
            axes[0].axvline(0.0, color="black", linestyle="--", linewidth=1.2)
            axes[0].set_title("SR - LR RMSE delta (lower is better for SR)")

            sns.barplot(data=top_delta, x="delta_r2_sr_minus_lr", y="config", ax=axes[1], errorbar=None)
            axes[1].axvline(0.0, color="black", linestyle="--", linewidth=1.2)
            axes[1].set_title("SR - LR R2 delta (higher is better for SR)")
            plt.tight_layout()
            fig.savefig(cmp_out / "sr_lr_delta_metrics.png", dpi=180, bbox_inches="tight")
            plt.show()

            if pred_frames:
                pred_all = pd.concat(pred_frames, ignore_index=True)
                pred_all.to_csv(cmp_out / "predictions_all_runs.csv", index=False)

                best_cfg = merged.iloc[0]
                mask_cfg = (
                    (pred_all["feature_set"] == best_cfg["feature_set"])
                    & (pred_all["selection_label"] == best_cfg["selection_label"])
                    & (pred_all["regressor"] == best_cfg["regressor"])
                )
                pred_cfg = pred_all[mask_cfg].copy()

                if not pred_cfg.empty and set(pred_cfg["resolution"].str.lower().unique()) >= {"sr", "lr"}:
                    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
                    for ax, res in zip(axes, ["lr", "sr"]):
                        sub = pred_cfg[pred_cfg["resolution"].str.lower() == res]
                        if sub.empty:
                            ax.set_title(res.upper() + " (missing)")
                            continue
                        ax.scatter(sub["y_true"], sub["y_pred"], alpha=0.6, s=16)
                        mn = float(min(sub["y_true"].min(), sub["y_pred"].min()))
                        mx = float(max(sub["y_true"].max(), sub["y_pred"].max()))
                        ax.plot([mn, mx], [mn, mx], "--", color="black", linewidth=1.1)
                        ax.set_title(res.upper())
                        ax.set_xlabel("Observation")
                        ax.set_ylabel("Prediction")
                    plt.tight_layout()
                    fig.savefig(cmp_out / "pred_vs_obs_best_config_sr_vs_lr.png", dpi=180, bbox_inches="tight")
                    plt.show()

            print(f"Saved SR/LR comparison artifacts to: {cmp_out}")

In [ ]:
# Backbone + band audit + raw-feature reduction sweep (LR-first, then SR)
from pathlib import Path
import json
import numpy as np
import subprocess
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from src.s2_pipeline.yellowness_model import build_yellowness_model
from src.s2_pipeline.yellowness_training import make_group_split

sns.set_theme(style="whitegrid", context="talk")

PYTHON_BIN = sys.executable
RUN_ON_KAGGLE = Path("/kaggle/working").exists()
REPO = globals().get("REPO", Path.cwd())
KAGGLE_DATASET_DIR = Path("/kaggle/input/datasets/ahmedtrabelsi88/yellow-beet")
DEFAULT_DATASET_DIR = KAGGLE_DATASET_DIR if RUN_ON_KAGGLE and KAGGLE_DATASET_DIR.exists() else REPO / "yellowness_dataset"
DEFAULT_INVENTORY = DEFAULT_DATASET_DIR / "observations_inventory.csv"
INVENTORY = Path(globals().get("INVENTORY", DEFAULT_INVENTORY))
DATASET_DIR = Path(globals().get("DATASET_DIR", DEFAULT_DATASET_DIR))
if RUN_ON_KAGGLE and KAGGLE_DATASET_DIR.exists():
    if (not INVENTORY.exists()) or ("yellowness_dataset" in str(INVENTORY).replace("\\", "/")):
        INVENTORY = DEFAULT_INVENTORY
    if (not DATASET_DIR.exists()) or ("yellowness_dataset" in str(DATASET_DIR).replace("\\", "/")):
        DATASET_DIR = DEFAULT_DATASET_DIR

print(f"Inventory path: {INVENTORY}")
print(f"Dataset root: {DATASET_DIR}")

SHAPEFILE_PATH = REPO / "data" / "shapefiles" / "2020_SEPIM.shp"
MASK_BUFFER_M = -20.0

# ------------------------------------------------------------------
# 1) Experiment grid
# ------------------------------------------------------------------
# Start with LR only for fast signal, then switch to ["lr", "sr"].
RUN_RESOLUTIONS = ["lr"]

BACKBONE_SPECS = [
    {
        "backbone": "torchgeo:resnet50",
        "weight": "ResNet50_Weights.SENTINEL2_ALL_NDVI_SECO_ECO",
        "short_name": "resnet50_ndvi_seco_eco",
    },
    {
        "backbone": "torchgeo:resnet50",
        "weight": "ResNet50_Weights.SENTINEL2_SI_MS_SATLAS",
        "short_name": "resnet50_satlas",
    },
    {
        "backbone": "torchgeo:resnet152",
        "weight": "ResNet152_Weights.SENTINEL2_SI_MS_SATLAS",
        "short_name": "resnet152_satlas",
    },
    {
        "backbone": "torchgeo:swin_v2_t",
        "weight": "Swin_V2_T_Weights.SENTINEL2_SI_MS_SATLAS",
        "short_name": "swin_v2_t_satlas",
    },
    {
        "backbone": "torchgeo:swin_v2_b",
        "weight": "Swin_V2_B_Weights.SENTINEL2_SI_MS_SATLAS",
        "short_name": "swin_v2_b_satlas",
    },
]

# Keep this compact first; expand only after finding promising backbones.
MASK_POOLING_CONFIGS = [
    {"mask_fusion": "feature_mask_pool", "pooling": "masked_avg", "tag": "feature_mask_pool_masked_avg"},
    {"mask_fusion": "feature_mask_pool", "pooling": "masked_mean_std", "tag": "feature_mask_pool_masked_mean_std"},
    {"mask_fusion": "image_only", "pooling": "masked_mean_std", "tag": "image_only_masked_mean_std"},
]

# Accepted by scripts/train_yellowness_backbone_ml.py
REGRESSORS = ["ridge", "pls", "random_forest", "extra_trees", "xgboost"]

# Raw feature reduction from frozen backbone vectors.
FEATURE_REDUCTION_GRID = [
    {"method": "none", "dim": 64, "variance": 0.95},
    {"method": "pca", "dim": 64, "variance": 0.95},
    {"method": "pca_var95", "dim": 64, "variance": 0.95},
    {"method": "mi_topk", "dim": 128, "variance": 0.95},
    {"method": "f_reg_topk", "dim": 128, "variance": 0.95},
    {"method": "var_topk", "dim": 128, "variance": 0.95},
]

# Optional caps for quick debugging.
MAX_BACKBONES = None   # e.g. 2
MAX_REDUCTION_CONFIGS = None  # e.g. 3

CENTER_CROP_SIZE = 128
INSPECT_RESOLUTION = "lr"
INSPECT_SAMPLE_INDEX = 0

OUT_ROOT = REPO / "outputs" / "both_resolutions_reduced_features"
OUT_ROOT.mkdir(parents=True, exist_ok=True)


def run_cmd(cmd, cwd=None):
    print("RUN:", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True, cwd=str(cwd) if cwd else None)


def expected_band_recipe(weight_name: str) -> str | None:
    satlas_9band_prefixes = (
        "ResNet50_Weights.SENTINEL2_SI_MS_SATLAS",
        "ResNet152_Weights.SENTINEL2_SI_MS_SATLAS",
        "ResNet50_Weights.SENTINEL2_MI_MS_SATLAS",
        "Swin_V2_T_Weights.SENTINEL2_SI_MS_SATLAS",
        "Swin_V2_T_Weights.SENTINEL2_MI_MS_SATLAS",
        "Swin_V2_B_Weights.SENTINEL2_SI_MS_SATLAS",
        "Swin_V2_B_Weights.SENTINEL2_MI_MS_SATLAS",
    )
    if weight_name.startswith(satlas_9band_prefixes):
        return "satlas_l1c_9_from_l2a"
    if "SENTINEL2_ALL_NDVI_SECO_ECO" in weight_name:
        return "ndvi_seco_eco_9"
    return None


def expected_image_channels(weight_name: str) -> int:
    recipe = expected_band_recipe(weight_name)
    return 9 if recipe in {"satlas_l1c_9_from_l2a", "ndvi_seco_eco_9"} else 10


backbones_active = BACKBONE_SPECS[:MAX_BACKBONES] if MAX_BACKBONES else BACKBONE_SPECS
reductions_active = FEATURE_REDUCTION_GRID[:MAX_REDUCTION_CONFIGS] if MAX_REDUCTION_CONFIGS else FEATURE_REDUCTION_GRID

band_audit_rows = []
for spec in backbones_active:
    recipe = expected_band_recipe(spec["weight"])
    band_audit_rows.append(
        {
            "backbone": spec["backbone"],
            "weight": spec["weight"],
            "short_name": spec["short_name"],
            "expected_band_recipe": recipe,
            "expected_image_channels": expected_image_channels(spec["weight"]),
        }
    )
band_audit_df = pd.DataFrame(band_audit_rows)
print("Planned backbone-band audit:")
display(band_audit_df)
band_audit_df.to_csv(OUT_ROOT / "planned_backbone_band_audit.csv", index=False)

# ------------------------------------------------------------------
# 2) Preflight inspect one sample with the first backbone
# ------------------------------------------------------------------
inspect_split = make_group_split(
    inventory_csv=INVENTORY,
    root_dir=DATASET_DIR,
    resolution=INSPECT_RESOLUTION,
    band_recipe=expected_band_recipe(backbones_active[0]["weight"]),
    shapefile_path=SHAPEFILE_PATH,
    shapefile_buffer_m=MASK_BUFFER_M,
)
inspect_sample = inspect_split.train[INSPECT_SAMPLE_INDEX]
inspect_image = inspect_sample["image"].numpy()
inspect_mask = inspect_sample["mask"].numpy()[0]
if CENTER_CROP_SIZE and CENTER_CROP_SIZE < inspect_image.shape[-1]:
    top = (inspect_image.shape[-2] - CENTER_CROP_SIZE) // 2
    left = (inspect_image.shape[-1] - CENTER_CROP_SIZE) // 2
    inspect_image = inspect_image[:, top:top + CENTER_CROP_SIZE, left:left + CENTER_CROP_SIZE]
    inspect_mask = inspect_mask[top:top + CENTER_CROP_SIZE, left:left + CENTER_CROP_SIZE]

mask_fraction = float(inspect_mask.mean())
print(
    f"Inspect sample resolution={INSPECT_RESOLUTION} | crop_size={CENTER_CROP_SIZE} | "
    f"image_shape={inspect_image.shape} | mask_shape={inspect_mask.shape} | mask_fraction={mask_fraction:.4f}"
)

rgb = np.clip(np.transpose(inspect_image[[2, 1, 0]], (1, 2, 0)), 0.0, 1.0)
masked_rgb = rgb * inspect_mask[..., None]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb)
axes[0].set_title("LR patch RGB")
axes[1].imshow(inspect_mask, cmap="gray")
axes[1].set_title("Parcel mask from file/.shp")
axes[2].imshow(masked_rgb)
axes[2].set_title("RGB masked by parcel geometry")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# 3) Run matrix: resolution x backbone x mask/pooling x raw reduction
# ------------------------------------------------------------------
summary_frames = []
audit_rows = []

total_runs = len(RUN_RESOLUTIONS) * len(backbones_active) * len(MASK_POOLING_CONFIGS) * len(reductions_active)
print(f"Planned runs: {total_runs}")

for resolution in RUN_RESOLUTIONS:
    for spec in backbones_active:
        backbone = spec["backbone"]
        weight = spec["weight"]
        short_name = spec["short_name"]
        recipe = expected_band_recipe(weight)
        channels = expected_image_channels(weight)

        for mask_cfg in MASK_POOLING_CONFIGS:
            for red_cfg in reductions_active:
                red_tag = f"{red_cfg['method']}_d{red_cfg['dim']}"
                run_name = f"{resolution}_{short_name}_{mask_cfg['tag']}_{red_tag}"
                out_dir = OUT_ROOT / run_name
                out_dir.mkdir(parents=True, exist_ok=True)

                cmd = [
                    PYTHON_BIN,
                    "scripts/train_yellowness_backbone_ml.py",
                    "--inventory", str(INVENTORY),
                    "--root-dir", str(DATASET_DIR),
                    "--resolution", resolution,
                    "--backbone", backbone,
                    "--torchgeo-weight", weight,
                    "--mask-fusion", mask_cfg["mask_fusion"],
                    "--pooling", mask_cfg["pooling"],
                    "--shapefile-path", str(SHAPEFILE_PATH),
                    "--mask-buffer-m", str(MASK_BUFFER_M),
                    "--feature-reduction-method", red_cfg["method"],
                    "--feature-reduction-dim", str(red_cfg["dim"]),
                    "--feature-reduction-variance", str(red_cfg["variance"]),
                    "--center-crop-size", str(CENTER_CROP_SIZE),
                    "--dr-methods", "none", "pca", "pls",
                    "--regressors", *REGRESSORS,
                    "--batch-size", "16",
                    "--num-workers", "4",
                    "--log-feature-shapes",
                    "--save-feature-csv",
                    "--feature-csv-name", f"{run_name}_embeddings.csv",
                    "--output-dir", str(out_dir),
                ]
                run_cmd(cmd, cwd=REPO)

                feature_cfg_path = out_dir / "feature_config.json"
                feature_cfg = {}
                if feature_cfg_path.exists():
                    feature_cfg = json.loads(feature_cfg_path.read_text(encoding="utf-8"))

                audit_rows.append(
                    {
                        "run_name": run_name,
                        "resolution": resolution,
                        "backbone": backbone,
                        "weight": weight,
                        "expected_band_recipe": recipe,
                        "expected_image_channels": channels,
                        "actual_band_recipe": feature_cfg.get("band_recipe"),
                        "actual_expected_image_channels": feature_cfg.get("expected_image_channels"),
                        "mask_fusion": mask_cfg["mask_fusion"],
                        "pooling": mask_cfg["pooling"],
                        "feature_reduction_method": red_cfg["method"],
                        "feature_reduction_dim": red_cfg["dim"],
                    }
                )

                summary_path = out_dir / "summary.csv"
                if summary_path.exists():
                    s = pd.read_csv(summary_path)
                    s["run_name"] = run_name
                    s["resolution"] = resolution
                    s["backbone_short"] = short_name
                    s["backbone"] = backbone
                    s["weight"] = weight
                    s["mask_fusion"] = mask_cfg["mask_fusion"]
                    s["pooling"] = mask_cfg["pooling"]
                    s["feature_reduction_method_cfg"] = red_cfg["method"]
                    s["feature_reduction_dim_cfg"] = red_cfg["dim"]
                    s["expected_band_recipe"] = recipe
                    s["expected_image_channels"] = channels
                    s["actual_band_recipe"] = feature_cfg.get("band_recipe")
                    s["actual_expected_image_channels"] = feature_cfg.get("expected_image_channels")
                    summary_frames.append(s)

if not summary_frames:
    raise RuntimeError("No summary.csv files were produced.")

summary_all = pd.concat(summary_frames, ignore_index=True)
summary_all = summary_all.sort_values(["val_rmse", "val_r2"], ascending=[True, False]).reset_index(drop=True)
summary_all.to_csv(OUT_ROOT / "summary_all_resolutions.csv", index=False)

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT_ROOT / "run_backbone_band_audit.csv", index=False)

print("Top results:")
display(summary_all.head(40))

print("Backbone/band mismatches (should be empty):")
mismatch = audit_df[
    (audit_df["actual_band_recipe"].fillna("") != audit_df["expected_band_recipe"].fillna(""))
    | (audit_df["actual_expected_image_channels"].fillna(-1).astype(int) != audit_df["expected_image_channels"].astype(int))
]
display(mismatch)

leaderboard = (
    summary_all.groupby(
        [
            "resolution",
            "backbone_short",
            "feature_reduction_method",
            "reduced_backbone_feature_dim",
            "mask_fusion",
            "pooling",
            "regressor",
        ],
        as_index=False,
    )[["val_rmse", "val_mae", "val_r2"]]
    .mean()
    .sort_values(["val_rmse", "val_r2"], ascending=[True, False])
    .reset_index(drop=True)
)
leaderboard.to_csv(OUT_ROOT / "leaderboard_by_backbone_reduction.csv", index=False)
print("Leaderboard (mean metrics):")
display(leaderboard.head(50))

# ------------------------------------------------------------------
# 4) Quick diagnostics plots
# ------------------------------------------------------------------
plot_df = leaderboard.head(min(30, len(leaderboard))).copy()
plot_df["config"] = plot_df.apply(
    lambda r: f"{r['backbone_short']} | {r['feature_reduction_method']} | {r['regressor']} | {r['mask_fusion']}:{r['pooling']}",
    axis=1,
)

fig, ax = plt.subplots(figsize=(20, max(7, 0.35 * len(plot_df))))
sns.barplot(data=plot_df, x="val_rmse", y="config", hue="resolution", ax=ax, errorbar=None)
ax.set_title("Top configurations by validation RMSE")
plt.tight_layout()
fig.savefig(OUT_ROOT / "top_configs_rmse.png", dpi=180, bbox_inches="tight")
plt.show()

red_view = (
    summary_all.groupby(["feature_reduction_method", "regressor"], as_index=False)[["val_rmse", "val_r2"]]
    .mean()
    .sort_values(["val_rmse", "val_r2"], ascending=[True, False])
)
fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(data=red_view, x="feature_reduction_method", y="val_rmse", hue="regressor", ax=ax, errorbar=None)
ax.set_title("Mean RMSE by raw-feature reduction method")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(OUT_ROOT / "reduction_method_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved combined outputs to: {OUT_ROOT}")